# Laptop Recommender System — Offline Training Pipeline

Follow these steps to train and save the models used by the web app.

1) Environment and data
- Ensure Python dependencies are installed from `requirements.txt`.
- Verify processed data exists in `data/cache/` and benchmark JSONs in `data/`.
- Optional: keep `top-active-reviewers.json` in project root to focus CF on active users.

2) Reproducibility
- This notebook seeds randomness for stable results.
- It logs model shapes on save/load to confirm trained artifacts are used.

3) Run order
- Run the next cell (“End-to-end training and saving pipeline”).
- Optional: run the evaluation cell to see NDCG.

4) Outputs
- Saved models:
  - `models/content.pkl`
  - `models/collaborative.pkl`
  - `models/hybrid.pkl`
- Restart the web app to load the new models.

5) Notes
- Preferences are applied as soft signals; the algorithms maintain their core purpose.
- If data or hyperparameters change, re-run this notebook.



## Troubleshooting & Tips

- If a Markdown cell shows a SyntaxError, it’s likely set to Code. Change the cell type to Markdown (press M) and run it again.
- If a code cell won’t accept shell commands, prefix them with `!` (e.g., `!pip install -r requirements.txt`).
- If the kernel isn’t running or no kernel is selected, choose a Python kernel from the toolbar and re-run all cells.
- To verify trained models are used in the app, check the server logs for matrix shapes at startup.
- Re-run this notebook whenever the dataset or hyperparameters change.



The first cell is used to retrain the model, it will load data, train, and save.

In [ ]:
# End-to-end training and saving pipeline (deterministic, reproducible)
import os
import random
import numpy as np

from Laptop_Recommender_System import create_laptop_recommender_system
from joblib import dump

# 1) Deterministic seeding
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# 2) Create recommender and load/clean data
recommender = create_laptop_recommender_system()
print("Loading and preprocessing data...")
df_laptop, df_rating = recommender.load_and_preprocess_data(force_reload=False)
print(f"Laptop shape: {df_laptop.shape}, Rating shape: {df_rating.shape}")

# 3) Initialize and train models (content-based + collaborative)
print("Initializing and training recommendation engines...")
recommender.initialize_recommendation_engines()

# Optional: ensure collaborative matrix factorization factors exist (SV D/NMF based on config)
try:
    recommender.collaborative_filter.fit_matrix_factorization(method='svd')
except Exception as e:
    print(f"Matrix factorization step skipped: {e}")

# 4) (Evaluation code removed as per project update)
# The evaluation module and related code have been removed from this project.

# 5) Save final trained models to models/ folder
os.makedirs('models', exist_ok=True)

# Content-based model
try:
    recommender.content_based_filter.save_model('models/content.pkl')
    print("Saved content-based model -> models/content.pkl")
except Exception as e:
    print(f"Failed to save content-based model: {e}")

# Collaborative model
try:
    recommender.collaborative_filter.save_model('models/collaborative.pkl')
    print("Saved collaborative model -> models/collaborative.pkl")
except Exception as e:
    print(f"Failed to save collaborative model: {e}")

# Hybrid: save a lightweight wrapper with weights and references needed
try:
    hybrid_payload = {
        'weights': recommender.config.get('hybrid', {}),
        'content_based_info': {
            'feature_matrix_shape': getattr(recommender.content_based_filter, 'feature_matrix', np.array([])).shape,
            'similarity_matrix_shape': getattr(recommender.content_based_filter, 'similarity_matrix', np.array([])).shape,
        },
        'collaborative_info': {
            'user_item_matrix_shape': getattr(recommender.collaborative_filter, 'user_item_matrix', np.array([[]])).shape,
        }
    }
    dump(hybrid_payload, 'models/hybrid.pkl', compress=3)
    print("Saved hybrid configuration -> models/hybrid.pkl")
except Exception as e:
    print(f"Failed to save hybrid configuration: {e}")

print("All tasks completed.")

🎯 Training Content-Based Filtering Model...
🚀 Starting Content-Based model training...
⏱️  Estimated time: 2-5 minutes (much faster now!)
🔄 Training with reviewer tier weighting...
🔄 Preparing features for content-based filtering...
🔄 Handling NaN values in features...
✅ Features prepared: (776, 1015)
✅ NaN values: 0
✅ Infinite values: 0
🔄 Calculating similarity matrix using cosine...
✅ Similarity matrix calculated: (776, 776)
🔄 Applying reviewer tier weighting (ultra-fast)...
  📊 Pre-calculating reviewer weights for each laptop...
  ✅ Calculated weights for 776 laptops
  🚀 Applying weights using vectorized operations...
✅ Model trained with reviewer weighting!
✅ Content-Based Filtering model trained successfully!

🧪 Testing Content-Based model...
Top 5 recommendations for laptop 0:
  1. Lenovo Legion Y740-15IRHg 81UH000GUS 15.6 Gaming N... (similarity: 0.584)
  2. LG gram (2022) 17Z90Q Ultra Lightweight Laptop, 17... (similarity: 0.477)
  3. ASUS TUF 17.3 FHD 144Hz IPS-Type Gaming Lap

Loading and preprocessing data...
Laptop shape: (776, 29), Rating shape: (13608, 14)
Initializing and training recommendation engines...


INFO:content_based_filtering:No trained content-based model found; will build from data
INFO:content_based_filtering:Created 31 brand features
INFO:content_based_filtering:Created 31 brand features
INFO:content_based_filtering:Created 31 brand features
INFO:content_based_filtering:Created 31 brand features
INFO:content_based_filtering:Feature matrix created with shape: (776, 2143)
INFO:content_based_filtering:Features used: ['text_0', 'text_1', 'text_2', 'text_3', 'text_4', 'text_5', 'text_6', 'text_7', 'text_8', 'text_9', 'text_10', 'text_11', 'text_12', 'text_13', 'text_14', 'text_15', 'text_16', 'text_17', 'text_18', 'text_19', 'text_20', 'text_21', 'text_22', 'text_23', 'text_24', 'text_25', 'text_26', 'text_27', 'text_28', 'text_29', 'text_30', 'text_31', 'text_32', 'text_33', 'text_34', 'text_35', 'text_36', 'text_37', 'text_38', 'text_39', 'text_40', 'text_41', 'text_42', 'text_43', 'text_44', 'text_45', 'text_46', 'text_47', 'text_48', 'text_49', 'text_50', 'text_51', 'text_52'

Evaluation skipped/not available: cannot import name 'evaluate_system' from 'evaluate_recommender_system' (C:\Users\hokk-\Lab Materials\Laptop Recommender System\evaluate_recommender_system.py)


INFO:content_based_filtering:Content-based model saved to models/content.pkl


Saved content-based model -> models/content.pkl


INFO:collaborative_filtering:Collaborative filtering model saved to models/collaborative.pkl


Saved collaborative model -> models/collaborative.pkl
Failed to save hybrid configuration: 'NoneType' object has no attribute 'shape'
All tasks completed.


This notebook provides a complete offline training pipeline for the laptop recommender system, including:
- **Data Loading and Preprocessing**: Load and clean the dataset with top active reviewers integration
- **Data Cleaning & Preparation**: Handle missing values, duplicates, encode categorical values, normalize features
- **Model Training**: Train Content-Based, Collaborative Filtering, and Hybrid models with parameter tuning
- **Model Evaluation**: Comprehensive evaluation using RMSE, MAE, NDCG
- **Model Serialization**: Save trained models as .pkl files in models/ folder for production use
- **Top Active Reviewers Integration**: Leverage reviewer expertise for enhanced recommendations
- **Web App Integration**: Prepare models for seamless integration with Flask web app

## Table of Contents
1. [Setup and Imports](#setup)
2. [Data Loading and Preprocessing](#data-loading)
3. [Top Active Reviewers Integration](#reviewers-integration)
4. [Data Cleaning & Preparation](#data-cleaning)
5. [Data Quality Assessment](#data-quality)
6. [Train/Test Split](#train-test-split)
7. [Content-Based Filtering Training](#content-based)
8. [Collaborative Filtering Training](#collaborative)
9. [Hybrid Model Training](#hybrid)
10. [Model Evaluation and Metrics](#evaluation)  <!-- Precision/Recall removed -->
11. [Model Serialization](#serialization)
12. [Database Mapping Creation](#database-mapping)
13. [Model Loading and Testing](#testing)
14. [Performance Report](#performance-report)

## 1. Setup and Imports {#setup}


In [2]:
# Import required libraries
import pandas as pd
import numpy as np
import pickle
import os
import json
from datetime import datetime
import logging
from typing import Dict, List, Optional, Tuple, Any
import warnings
warnings.filterwarnings('ignore')

# Machine Learning libraries
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
from scipy.spatial.distance import pdist, squareform

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Progress tracking
from tqdm import tqdm

# Import our custom modules
from Laptop_Recommender_System import LaptopRecommenderSystem
from content_based_filtering import ContentBasedFiltering
from collaborative_filtering import CollaborativeFiltering
from data_preprocessing import LaptopDataPreprocessor

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✅ All imports successful!")
print(f"📅 Training started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


✅ All imports successful!
📅 Training started at: 2025-09-07 21:13:53


## 2. Data Loading and Preprocessing {#data-loading}


In [3]:
# Initialize the data preprocessor
print("🔄 Initializing data preprocessor...")
preprocessor = LaptopDataPreprocessor()

# Load and preprocess data
print("📊 Loading and preprocessing data...")
df_laptop, df_rating = preprocessor.preprocess_separated_pipeline()

print(f"✅ Data loaded successfully!")
print(f"📱 Laptop data shape: {df_laptop.shape}")
print(f"⭐ Rating data shape: {df_rating.shape}")

# Display basic information about the datasets
print("\n📋 Dataset Summary:")
print(f"• Total laptops: {len(df_laptop)}")
print(f"• Total ratings: {len(df_rating)}")
print(f"• Unique users: {df_rating['user_id_encoded'].nunique() if 'user_id_encoded' in df_rating.columns else 'N/A'}")
print(f"• Unique laptops: {df_laptop['asin'].nunique()}")
print(f"• Average rating: {df_laptop['average_rating'].mean():.2f}")
print(f"• Price range: RM {df_laptop['price_myr'].min():.2f} - RM {df_laptop['price_myr'].max():.2f}")


INFO:data_preprocessing:Loaded cached data: 776 laptops, 13608 ratings
INFO:data_preprocessing:Using cached preprocessed data


🔄 Initializing data preprocessor...
📊 Loading and preprocessing data...
✅ Data loaded successfully!
📱 Laptop data shape: (776, 29)
⭐ Rating data shape: (13608, 14)

📋 Dataset Summary:
• Total laptops: 776
• Total ratings: 13608
• Unique users: 13292
• Unique laptops: 776
• Average rating: 4.07
• Price range: RM 1850.41 - RM 13299.76


## 3. Top Active Reviewers Integration {#reviewers-integration}


In [4]:
# Load top active reviewers data
print("🔍 Loading top active reviewers data...")
with open('top-active-reviewers.json', 'r') as f:
    top_reviewers = json.load(f)

print(f"✅ Loaded {len(top_reviewers)} top active reviewers")

# Convert to DataFrame for easier manipulation
reviewers_df = pd.DataFrame(top_reviewers)
print(f"📊 Top reviewers rating counts: {reviewers_df['rating_count'].describe()}")

# Create a mapping of user_id to reviewer tier
reviewer_tiers = {}
for i, reviewer in enumerate(top_reviewers):
    if reviewer['rating_count'] >= 10:
        reviewer_tiers[reviewer['user_id']] = 'expert'
    elif reviewer['rating_count'] >= 6:
        reviewer_tiers[reviewer['user_id']] = 'experienced'
    elif reviewer['rating_count'] >= 4:
        reviewer_tiers[reviewer['user_id']] = 'active'
    else:
        reviewer_tiers[reviewer['user_id']] = 'regular'

print(f"📈 Reviewer tier distribution:")
tier_counts = pd.Series(list(reviewer_tiers.values())).value_counts()
for tier, count in tier_counts.items():
    print(f"  • {tier}: {count} reviewers")

# Add reviewer tier to rating data
print("🔄 Adding reviewer tiers to rating data...")
df_rating['reviewer_tier'] = df_rating['user_id_encoded'].map(reviewer_tiers).fillna('casual')
print(f"✅ Added reviewer tiers. Distribution:")
print(df_rating['reviewer_tier'].value_counts())


🔍 Loading top active reviewers data...
✅ Loaded 126 top active reviewers
📊 Top reviewers rating counts: count    126.000000
mean       3.476190
std        0.993694
min        3.000000
25%        3.000000
50%        3.000000
75%        4.000000
max       11.000000
Name: rating_count, dtype: float64
📈 Reviewer tier distribution:
  • regular: 89 reviewers
  • active: 34 reviewers
  • experienced: 2 reviewers
  • expert: 1 reviewers
🔄 Adding reviewer tiers to rating data...
✅ Added reviewer tiers. Distribution:
reviewer_tier
casual     13426
regular      119
active        55
expert         8
Name: count, dtype: int64


## 4. Data Cleaning & Preparation {#data-cleaning}


In [5]:
# Comprehensive Data Cleaning and Preparation
print("🧹 Starting comprehensive data cleaning and preparation...")

# 1. Handle Missing Values
print("\n1️⃣ Handling missing values...")
print("Laptop data missing values:")
print(df_laptop.isnull().sum().sort_values(ascending=False))

print("\nRating data missing values:")
print(df_rating.isnull().sum().sort_values(ascending=False))

# Fill missing values in laptop data
df_laptop['features_clean'] = df_laptop['features_clean'].fillna('')
df_laptop['title_y_clean'] = df_laptop['title_y_clean'].fillna('')
df_laptop['processor_model'] = df_laptop['processor_model'].fillna('Unknown')
df_laptop['gpu_model'] = df_laptop['gpu_model'].fillna('Unknown')

# Fill missing values in rating data
df_rating['text_clean'] = df_rating['text_clean'].fillna('')
df_rating['title_x_clean'] = df_rating['title_x_clean'].fillna('')

print("✅ Missing values handled")

# 2. Handle Duplicates
print("\n2️⃣ Handling duplicates...")
laptop_duplicates = df_laptop.duplicated(subset=['asin']).sum()
rating_duplicates = df_rating.duplicated(subset=['asin', 'user_id_encoded']).sum()

print(f"Laptop duplicates: {laptop_duplicates}")
print(f"Rating duplicates: {rating_duplicates}")

# Remove duplicates
df_laptop = df_laptop.drop_duplicates(subset=['asin'])
df_rating = df_rating.drop_duplicates(subset=['asin', 'user_id_encoded'])

print("✅ Duplicates removed")

# 3. Handle Inconsistent Labels
print("\n3️⃣ Handling inconsistent labels...")
print("Brand distribution:")
print(df_laptop['brand_original'].value_counts().head(10))

# Standardize brand names
brand_mapping = {
    'ASUS': 'Asus',
    'DELL': 'Dell',
    'HP': 'HP',
    'LENOVO': 'Lenovo',
    'ACER': 'Acer',
    'MSI': 'MSI',
    'APPLE': 'Apple',
    'SAMSUNG': 'Samsung'
}

df_laptop['brand_standardized'] = df_laptop['brand_original'].str.upper().map(brand_mapping).fillna(df_laptop['brand_original'])
print("✅ Brand names standardized")

# 4. Encode Categorical Values
print("\n4️⃣ Encoding categorical values...")

# Create label encoders for categorical features
categorical_features = ['brand_standardized', 'processor_model', 'gpu_model', 'storage_type', 'ram_type']
label_encoders = {}

for feature in categorical_features:
    if feature in df_laptop.columns:
        # Fill any remaining NaN values with 'Unknown'
        if df_laptop[feature].isna().sum() > 0:
            print(f"  {feature}: Filling {df_laptop[feature].isna().sum()} NaN values with 'Unknown'...")
            df_laptop[feature] = df_laptop[feature].fillna('Unknown')
        
        le = LabelEncoder()
        df_laptop[f'{feature}_encoded'] = le.fit_transform(df_laptop[feature].astype(str))
        label_encoders[feature] = le
        print(f"✅ Encoded {feature}: {len(le.classes_)} unique values")

# 5. Normalize Numerical Features
print("\n5️⃣ Normalizing numerical features...")

numerical_features = ['price_myr', 'ram_gb', 'storage_gb', 'screen_size_inches', 
                     'cpu_benchmark_score', 'gpu_benchmark_score', 'total_benchmark_score']

# Check for NaN values before normalization
print("Checking for NaN values in numerical features:")
for feature in numerical_features:
    nan_count = df_laptop[feature].isna().sum()
    if nan_count > 0:
        print(f"  {feature}: {nan_count} NaN values found, filling with median...")
        df_laptop[feature] = df_laptop[feature].fillna(df_laptop[feature].median())

# Check for infinite values
print("Checking for infinite values in numerical features:")
for feature in numerical_features:
    inf_count = np.isinf(df_laptop[feature]).sum()
    if inf_count > 0:
        print(f"  {feature}: {inf_count} infinite values found, replacing with median...")
        df_laptop[feature] = df_laptop[feature].replace([np.inf, -np.inf], df_laptop[feature].median())

scaler = StandardScaler()
df_laptop[numerical_features] = scaler.fit_transform(df_laptop[numerical_features])

print("✅ Numerical features normalized")

# 6. Create Enhanced Features
print("\n6️⃣ Creating enhanced features...")

# Price per performance ratio (handle division by zero)
df_laptop['price_performance_ratio'] = df_laptop['price_myr'] / (df_laptop['total_benchmark_score'] + 1)

# Gaming capability score
df_laptop['gaming_score'] = (df_laptop['gpu_benchmark_score'] * 0.7 + 
                            df_laptop['cpu_benchmark_score'] * 0.3)

# Productivity score
df_laptop['productivity_score'] = (df_laptop['cpu_benchmark_score'] * 0.6 + 
                                  df_laptop['ram_gb'] * 0.4)

# Handle any NaN or infinite values in enhanced features
enhanced_features = ['price_performance_ratio', 'gaming_score', 'productivity_score']
for feature in enhanced_features:
    if df_laptop[feature].isna().sum() > 0:
        print(f"  {feature}: Filling NaN values with median...")
        df_laptop[feature] = df_laptop[feature].fillna(df_laptop[feature].median())
    
    if np.isinf(df_laptop[feature]).sum() > 0:
        print(f"  {feature}: Replacing infinite values with median...")
        df_laptop[feature] = df_laptop[feature].replace([np.inf, -np.inf], df_laptop[feature].median())

print("✅ Enhanced features created")

# 7. Text Processing for Content-Based Filtering
print("\n7️⃣ Processing text features for content-based filtering...")

# Combine title and features for TF-IDF
df_laptop['combined_text'] = (df_laptop['title_y_clean'] + ' ' + 
                             df_laptop['features_clean'] + ' ' + 
                             df_laptop['processor_model'].astype(str) + ' ' + 
                             df_laptop['gpu_model'].astype(str))

# Initialize TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=1000,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2
)

# Fit and transform the combined text
tfidf_matrix = tfidf_vectorizer.fit_transform(df_laptop['combined_text'])
print(f"✅ TF-IDF matrix created: {tfidf_matrix.shape}")

print("\n🎉 Data cleaning and preparation completed!")
print(f"Final laptop data shape: {df_laptop.shape}")
print(f"Final rating data shape: {df_rating.shape}")


🧹 Starting comprehensive data cleaning and preparation...

1️⃣ Handling missing values...
Laptop data missing values:
ram_type                 217
storage_gb                42
screen_size_inches        37
storage_type              26
processor_model            2
ram_gb                     1
brand_encoded              0
brand_original             0
asin                       0
parent_asin                0
price_usd                  0
price_myr                  0
price_category_myr         0
average_rating             0
features_clean             0
title_y_clean              0
store_encoded              0
color_encoded              0
os_encoded                 0
gpu_model                  0
rating_number              0
gaming_capability          0
performance_tier           0
total_benchmark_score      0
gpu_benchmark_score        0
cpu_benchmark_score        0
images_y                   0
videos                     0
laptop_id                  0
dtype: int64

Rating data missing values:

## 5. Data Quality Assessment {#data-quality}


In [6]:
# Comprehensive Data Quality Assessment
print("📊 Performing comprehensive data quality assessment...")

# 1. Dataset Statistics
print("\n1️⃣ Dataset Statistics:")
print(f"Laptop Dataset:")
print(f"  • Shape: {df_laptop.shape}")
print(f"  • Memory usage: {df_laptop.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"  • Columns: {list(df_laptop.columns)}")

print(f"\nRating Dataset:")
print(f"  • Shape: {df_rating.shape}")
print(f"  • Memory usage: {df_rating.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"  • Columns: {list(df_rating.columns)}")

# 2. Rating Distribution Analysis
print("\n2️⃣ Rating Distribution Analysis:")
rating_dist = df_rating['rating'].value_counts().sort_index()
print("Rating distribution:")
for rating, count in rating_dist.items():
    percentage = (count / len(df_rating)) * 100
    print(f"  • {rating} stars: {count} ({percentage:.1f}%)")

# 3. User Activity Analysis
print("\n3️⃣ User Activity Analysis:")
user_activity = df_rating.groupby('user_id_encoded').size()
print(f"User rating statistics:")
print(f"  • Mean ratings per user: {user_activity.mean():.2f}")
print(f"  • Median ratings per user: {user_activity.median():.2f}")
print(f"  • Max ratings per user: {user_activity.max()}")
print(f"  • Users with 1 rating: {(user_activity == 1).sum()}")
print(f"  • Users with 5+ ratings: {(user_activity >= 5).sum()}")

# 4. Laptop Popularity Analysis
print("\n4️⃣ Laptop Popularity Analysis:")
laptop_popularity = df_rating.groupby('asin').size()
print(f"Laptop rating statistics:")
print(f"  • Mean ratings per laptop: {laptop_popularity.mean():.2f}")
print(f"  • Median ratings per laptop: {laptop_popularity.median():.2f}")
print(f"  • Max ratings per laptop: {laptop_popularity.max()}")
print(f"  • Laptops with 1 rating: {(laptop_popularity == 1).sum()}")
print(f"  • Laptops with 10+ ratings: {(laptop_popularity >= 10).sum()}")

# 5. Reviewer Tier Impact Analysis
print("\n5️⃣ Reviewer Tier Impact Analysis:")
tier_analysis = df_rating.groupby('reviewer_tier').agg({
    'rating': ['count', 'mean', 'std'],
    'helpful_vote': 'mean'
}).round(2)
print("Reviewer tier analysis:")
print(tier_analysis)

# 6. Data Sparsity Analysis
print("\n6️⃣ Data Sparsity Analysis:")
total_possible_ratings = len(df_laptop) * len(df_rating['user_id_encoded'].unique())
actual_ratings = len(df_rating)
sparsity = 1 - (actual_ratings / total_possible_ratings)
print(f"  • Total possible ratings: {total_possible_ratings:,}")
print(f"  • Actual ratings: {actual_ratings:,}")
print(f"  • Sparsity: {sparsity:.4f} ({sparsity*100:.2f}%)")

# 7. Temporal Analysis
print("\n7️⃣ Temporal Analysis:")
if 'timestamp' in df_rating.columns:
    df_rating['timestamp'] = pd.to_datetime(df_rating['timestamp'])
    print(f"  • Date range: {df_rating['timestamp'].min()} to {df_rating['timestamp'].max()}")
    print(f"  • Ratings per year:")
    yearly_ratings = df_rating.groupby(df_rating['timestamp'].dt.year).size()
    for year, count in yearly_ratings.items():
        print(f"    - {year}: {count} ratings")

print("\n✅ Data quality assessment completed!")


📊 Performing comprehensive data quality assessment...

1️⃣ Dataset Statistics:
Laptop Dataset:
  • Shape: (776, 39)
  • Memory usage: 1.52 MB
  • Columns: ['asin', 'parent_asin', 'price_usd', 'price_myr', 'price_category_myr', 'brand_original', 'brand_encoded', 'os_encoded', 'color_encoded', 'store_encoded', 'title_y_clean', 'features_clean', 'average_rating', 'rating_number', 'processor_model', 'gpu_model', 'cpu_benchmark_score', 'gpu_benchmark_score', 'total_benchmark_score', 'performance_tier', 'gaming_capability', 'ram_gb', 'storage_gb', 'screen_size_inches', 'storage_type', 'ram_type', 'images_y', 'videos', 'laptop_id', 'brand_standardized', 'brand_standardized_encoded', 'processor_model_encoded', 'gpu_model_encoded', 'storage_type_encoded', 'ram_type_encoded', 'price_performance_ratio', 'gaming_score', 'productivity_score', 'combined_text']

Rating Dataset:
  • Shape: (13446, 15)
  • Memory usage: 8.52 MB
  • Columns: ['asin', 'parent_asin', 'user_id_encoded', 'timestamp', 'ratin

## 6. Train/Test Split {#train-test-split}


In [7]:
# Train/Test Split for Recommendation Systems
print("🔄 Creating train/test split for recommendation systems...")

# For collaborative filtering, we need to split ratings while ensuring each user and item appears in both sets
def create_train_test_split(df_rating, test_size=0.2, random_state=42):
    """
    Create train/test split ensuring each user and item appears in both sets
    """
    np.random.seed(random_state)
    
    # Group by user to ensure each user has ratings in both train and test
    train_ratings = []
    test_ratings = []
    
    for user_id, user_ratings in df_rating.groupby('user_id_encoded'):
        if len(user_ratings) >= 2:  # Only users with 2+ ratings
            # Randomly select one rating for test set
            test_idx = np.random.choice(len(user_ratings), 1, replace=False)
            test_ratings.append(user_ratings.iloc[test_idx])
            train_ratings.append(user_ratings.drop(user_ratings.index[test_idx]))
        else:
            # Users with only 1 rating go to training set
            train_ratings.append(user_ratings)
    
    train_df = pd.concat(train_ratings, ignore_index=True)
    test_df = pd.concat(test_ratings, ignore_index=True)
    
    return train_df, test_df

# Create train/test split
train_ratings, test_ratings = create_train_test_split(df_rating, test_size=0.2, random_state=42)

print(f"✅ Train/test split completed:")
print(f"  • Training set: {len(train_ratings)} ratings")
print(f"  • Test set: {len(test_ratings)} ratings")
print(f"  • Train ratio: {len(train_ratings) / len(df_rating):.2%}")

# Verify that all users and items appear in both sets
train_users = set(train_ratings['user_id_encoded'].unique())
test_users = set(test_ratings['user_id_encoded'].unique())
train_items = set(train_ratings['asin'].unique())
test_items = set(test_ratings['asin'].unique())

print(f"\n📊 Split validation:")
print(f"  • Users in train: {len(train_users)}")
print(f"  • Users in test: {len(test_users)}")
print(f"  • Users in both: {len(train_users.intersection(test_users))}")
print(f"  • Items in train: {len(train_items)}")
print(f"  • Items in test: {len(test_items)}")
print(f"  • Items in both: {len(train_items.intersection(test_items))}")

# Create user-item matrices for collaborative filtering
def create_user_item_matrix(df, user_col='user_id_encoded', item_col='asin', rating_col='rating'):
    """Create user-item rating matrix"""
    matrix = df.pivot_table(
        index=user_col, 
        columns=item_col, 
        values=rating_col, 
        fill_value=0
    )
    return matrix

# Create matrices
train_matrix = create_user_item_matrix(train_ratings)
test_matrix = create_user_item_matrix(test_ratings)

print(f"\n📈 User-item matrices created:")
print(f"  • Train matrix shape: {train_matrix.shape}")
print(f"  • Test matrix shape: {test_matrix.shape}")
print(f"  • Train matrix sparsity: {(train_matrix == 0).sum().sum() / (train_matrix.shape[0] * train_matrix.shape[1]):.4f}")

print("✅ Data prepared for model training!")


🔄 Creating train/test split for recommendation systems...
✅ Train/test split completed:
  • Training set: 13315 ratings
  • Test set: 131 ratings
  • Train ratio: 99.03%

📊 Split validation:
  • Users in train: 13292
  • Users in test: 131
  • Users in both: 131
  • Items in train: 773
  • Items in test: 83
  • Items in both: 80

📈 User-item matrices created:
  • Train matrix shape: (13292, 773)
  • Test matrix shape: (131, 83)
  • Train matrix sparsity: 0.9987
✅ Data prepared for model training!


## 7. Evaluation Metrics Implementation {#evaluation-metrics}


In [8]:
# Comprehensive Evaluation Metrics Implementation
print("📊 Implementing comprehensive evaluation metrics...")

class RecommendationEvaluator:
    """Comprehensive evaluation metrics for recommendation systems"""
    
    def __init__(self):
        self.metrics = {}
    
    def calculate_rmse(self, y_true, y_pred):
        """Calculate Root Mean Square Error"""
        return np.sqrt(mean_squared_error(y_true, y_pred))
    
    def calculate_mae(self, y_true, y_pred):
        """Calculate Mean Absolute Error"""
        return mean_absolute_error(y_true, y_pred)
    
    def calculate_precision_at_k(self, y_true, y_pred, k=10):
        """Calculate Precision@K"""
        # Get top-k predictions
        top_k_indices = np.argsort(y_pred)[-k:]
        # Count relevant items in top-k
        relevant_items = np.sum(y_true[top_k_indices] >= 4)  # Assuming 4+ is relevant
        return relevant_items / k
    
    def calculate_recall_at_k(self, y_true, y_pred, k=10):
        """Calculate Recall@K"""
        # Get top-k predictions
        top_k_indices = np.argsort(y_pred)[-k:]
        # Count relevant items in top-k
        relevant_in_top_k = np.sum(y_true[top_k_indices] >= 4)
        # Total relevant items
        total_relevant = np.sum(y_true >= 4)
        return relevant_in_top_k / total_relevant if total_relevant > 0 else 0
    
    def calculate_ndcg_at_k(self, y_true, y_pred, k=10):
        """Calculate Normalized Discounted Cumulative Gain@K"""
        # Get top-k predictions
        top_k_indices = np.argsort(y_pred)[-k:]
        # Calculate DCG
        dcg = 0
        for i, idx in enumerate(top_k_indices):
            if y_true[idx] >= 4:  # Assuming 4+ is relevant
                dcg += (2**y_true[idx] - 1) / np.log2(i + 2)
        
        # Calculate IDCG (ideal DCG)
        ideal_scores = np.sort(y_true)[::-1][:k]
        idcg = 0
        for i, score in enumerate(ideal_scores):
            if score >= 4:
                idcg += (2**score - 1) / np.log2(i + 2)
        
        return dcg / idcg if idcg > 0 else 0
    
    def calculate_map_at_k(self, y_true, y_pred, k=10):
        """Calculate Mean Average Precision@K"""
        # Get top-k predictions
        top_k_indices = np.argsort(y_pred)[-k:]
        # Calculate precision at each position
        precision_sum = 0
        relevant_count = 0
        
        for i, idx in enumerate(top_k_indices):
            if y_true[idx] >= 4:  # Assuming 4+ is relevant
                relevant_count += 1
                precision_sum += relevant_count / (i + 1)
        
        return precision_sum / min(k, np.sum(y_true >= 4)) if np.sum(y_true >= 4) > 0 else 0
    
    def evaluate_model(self, y_true, y_pred, k_values=[5, 10, 20]):
        """Comprehensive model evaluation"""
        results = {}
        
        # Regression metrics
        results['RMSE'] = self.calculate_rmse(y_true, y_pred)
        results['MAE'] = self.calculate_mae(y_true, y_pred)
        
        # Ranking metrics for different k values
        for k in k_values:
            results[f'Precision@{k}'] = self.calculate_precision_at_k(y_true, y_pred, k)
            results[f'Recall@{k}'] = self.calculate_recall_at_k(y_true, y_pred, k)
            results[f'NDCG@{k}'] = self.calculate_ndcg_at_k(y_true, y_pred, k)
            results[f'MAP@{k}'] = self.calculate_map_at_k(y_true, y_pred, k)
        
        return results
    
    def evaluate_collaborative_filtering(self, model, test_matrix, k_values=[5, 10, 20]):
        """Evaluate collaborative filtering model using scikit-learn"""
        # For SVD models, calculate reconstruction error
        if hasattr(model, 'transform'):
            transformed = model.transform(test_matrix)
            reconstructed = model.inverse_transform(transformed)
            y_true = test_matrix.values.flatten()
            y_pred = reconstructed.flatten()
            
            # Calculate metrics
            results = self.evaluate_model(y_true, y_pred, k_values)
            results['Reconstruction_RMSE'] = self.calculate_rmse(y_true, y_pred)
            results['Reconstruction_MAE'] = self.calculate_mae(y_true, y_pred)
        else:
            # For KNN models, use a simple evaluation
            results = {'Model_Type': 'KNN', 'Score': 'Neighbor-based evaluation'}
        
        return results
    
    def evaluate_content_based(self, similarity_matrix, test_ratings, k_values=[5, 10, 20]):
        """Evaluate content-based filtering model"""
        results = {}
        
        # For each user in test set, get their actual ratings and predictions
        user_metrics = []
        
        for user_id in test_ratings['user_id_encoded'].unique():
            user_test_ratings = test_ratings[test_ratings['user_id_encoded'] == user_id]
            
            if len(user_test_ratings) > 0:
                # Get user's actual ratings
                actual_ratings = user_test_ratings['rating'].values
                
                # Get predictions (simplified - using average similarity)
                predicted_ratings = np.full(len(actual_ratings), 3.5)  # Placeholder
                
                # Calculate metrics for this user
                user_result = self.evaluate_model(actual_ratings, predicted_ratings, k_values)
                user_metrics.append(user_result)
        
        # Average metrics across all users
        if user_metrics:
            for metric in user_metrics[0].keys():
                results[metric] = np.mean([user_metric[metric] for user_metric in user_metrics])
        
        return results

# Initialize evaluator
evaluator = RecommendationEvaluator()
print("✅ Evaluation metrics implemented!")

# Test the evaluator with sample data
print("\n🧪 Testing evaluation metrics with sample data...")
sample_true = np.array([5, 4, 3, 2, 1, 5, 4, 3, 2, 1])
sample_pred = np.array([4.5, 3.8, 3.2, 2.1, 1.2, 4.7, 3.9, 2.8, 2.3, 1.1])

sample_results = evaluator.evaluate_model(sample_true, sample_pred, k_values=[5, 10])
print("Sample evaluation results:")
for metric, value in sample_results.items():
    print(f"  • {metric}: {value:.4f}")

print("✅ Evaluation metrics tested successfully!")


📊 Implementing comprehensive evaluation metrics...
✅ Evaluation metrics implemented!

🧪 Testing evaluation metrics with sample data...
Sample evaluation results:
  • RMSE: 0.2490
  • MAE: 0.2200
  • Precision@5: 0.8000
  • Recall@5: 1.0000
  • NDCG@5: 0.6557
  • MAP@5: 0.6792
  • Precision@10: 0.4000
  • Recall@10: 1.0000
  • NDCG@10: 0.4344
  • MAP@10: 0.2815
✅ Evaluation metrics tested successfully!


## 8. Content-Based Filtering Training {#content-based}


In [9]:
# Content-Based Filtering Model Training
print("🎯 Training Content-Based Filtering Model...")

class ContentBasedFiltering:
    """Enhanced Content-Based Filtering with reviewer tier weighting"""
    
    def __init__(self):
        self.similarity_matrix = None
        self.laptop_features = None
        self.tfidf_vectorizer = None
        self.scaler = None
        
    def prepare_features(self, df_laptop, reviewer_tiers=None):
        """Prepare features for content-based filtering"""
        print("🔄 Preparing features for content-based filtering...")
        
        # Combine text features
        df_laptop['combined_text'] = (
            df_laptop['title_y_clean'] + ' ' + 
            df_laptop['features_clean'] + ' ' + 
            df_laptop['processor_model'].astype(str) + ' ' + 
            df_laptop['gpu_model'].astype(str)
        )
        
        # TF-IDF for text features
        self.tfidf_vectorizer = TfidfVectorizer(
            max_features=1000,
            stop_words='english',
            ngram_range=(1, 2),
            min_df=2
        )
        
        tfidf_features = self.tfidf_vectorizer.fit_transform(df_laptop['combined_text'])
        
        # Numerical features
        numerical_features = [
            'price_myr', 'ram_gb', 'storage_gb', 'screen_size_inches',
            'cpu_benchmark_score', 'gpu_benchmark_score', 'total_benchmark_score',
            'gaming_score', 'productivity_score', 'price_performance_ratio'
        ]
        
        # Categorical features (one-hot encoded)
        categorical_features = [
            'brand_standardized_encoded', 'processor_model_encoded', 
            'gpu_model_encoded', 'storage_type_encoded', 'ram_type_encoded'
        ]
        
        # Combine all features
        feature_list = []
        
        # Add TF-IDF features
        feature_list.append(tfidf_features)
        
        # Add numerical features
        numerical_data = df_laptop[numerical_features].values
        feature_list.append(numerical_data)
        
        # Add categorical features
        categorical_data = df_laptop[categorical_features].values
        feature_list.append(categorical_data)
        
        # Combine all features
        self.laptop_features = np.hstack([
            tfidf_features.toarray(),
            numerical_data,
            categorical_data
        ])
        
        # Handle NaN values
        print("🔄 Handling NaN values in features...")
        nan_count_before = np.isnan(self.laptop_features).sum()
        if nan_count_before > 0:
            print(f"  Found {nan_count_before} NaN values, replacing with 0...")
            self.laptop_features = np.nan_to_num(self.laptop_features, nan=0.0, posinf=0.0, neginf=0.0)
        
        # Ensure all values are finite
        if not np.all(np.isfinite(self.laptop_features)):
            print("  Replacing infinite values with 0...")
            self.laptop_features = np.nan_to_num(self.laptop_features, nan=0.0, posinf=0.0, neginf=0.0)
        
        print(f"✅ Features prepared: {self.laptop_features.shape}")
        print(f"✅ NaN values: {np.isnan(self.laptop_features).sum()}")
        print(f"✅ Infinite values: {np.isinf(self.laptop_features).sum()}")
        return self.laptop_features
    
    def calculate_similarity(self, method='cosine'):
        """Calculate similarity matrix between laptops"""
        print(f"🔄 Calculating similarity matrix using {method}...")
        
        if method == 'cosine':
            self.similarity_matrix = cosine_similarity(self.laptop_features)
        elif method == 'euclidean':
            distances = pdist(self.laptop_features, metric='euclidean')
            self.similarity_matrix = 1 / (1 + squareform(distances))
        elif method == 'manhattan':
            distances = pdist(self.laptop_features, metric='cityblock')
            self.similarity_matrix = 1 / (1 + squareform(distances))
        
        print(f"✅ Similarity matrix calculated: {self.similarity_matrix.shape}")
        return self.similarity_matrix
    
    def get_recommendations(self, laptop_id, n_recommendations=10, exclude_rated=True, user_ratings=None):
        """Get recommendations for a laptop"""
        if self.similarity_matrix is None:
            raise ValueError("Similarity matrix not calculated. Run calculate_similarity() first.")
        
        # Get similarity scores for the laptop
        similarity_scores = self.similarity_matrix[laptop_id]
        
        # Exclude the laptop itself
        similarity_scores[laptop_id] = 0
        
        # Exclude already rated laptops if user_ratings provided
        if exclude_rated and user_ratings is not None:
            rated_laptops = user_ratings['asin'].values
            for rated_laptop in rated_laptops:
                if rated_laptop in df_laptop['asin'].values:
                    rated_idx = df_laptop[df_laptop['asin'] == rated_laptop].index[0]
                    similarity_scores[rated_idx] = 0
        
        # Get top recommendations
        top_indices = np.argsort(similarity_scores)[-n_recommendations:][::-1]
        
        recommendations = []
        for idx in top_indices:
            if similarity_scores[idx] > 0:
                recommendations.append({
                    'laptop_id': idx,
                    'asin': df_laptop.iloc[idx]['asin'],
                    'title': df_laptop.iloc[idx]['title_y_clean'],
                    'similarity_score': similarity_scores[idx],
                    'price': df_laptop.iloc[idx]['price_myr'],
                    'rating': df_laptop.iloc[idx]['average_rating']
                })
        
        return recommendations
    
    def train_with_reviewer_weighting(self, df_laptop, df_rating, reviewer_tiers):
        """Train model with reviewer tier weighting (ultra-fast version)"""
        print("🔄 Training with reviewer tier weighting...")
        
        # Prepare features
        self.prepare_features(df_laptop)
        
        # Calculate base similarity
        self.calculate_similarity()
        
        # Apply reviewer tier weighting (ultra-fast approach)
        print("🔄 Applying reviewer tier weighting (ultra-fast)...")
        tier_weights = {
            'expert': 1.5,
            'experienced': 1.2,
            'active': 1.0,
            'regular': 0.8,
            'casual': 0.6
        }
        
        # Pre-calculate reviewer tier weights for each laptop (much faster)
        print("  📊 Pre-calculating reviewer weights for each laptop...")
        laptop_reviewer_weights = {}
        
        for asin in df_laptop['asin']:
            laptop_ratings = df_rating[df_rating['asin'] == asin]
            if len(laptop_ratings) > 0:
                avg_weight = laptop_ratings['reviewer_tier'].map(tier_weights).mean()
                laptop_reviewer_weights[asin] = avg_weight
            else:
                laptop_reviewer_weights[asin] = 1.0  # Default weight
        
        print(f"  ✅ Calculated weights for {len(laptop_reviewer_weights)} laptops")
        
        # Ultra-fast vectorized approach - no nested loops!
        print("  🚀 Applying weights using vectorized operations...")
        
        # Create weight vector for all laptops
        weight_vector = np.array([laptop_reviewer_weights.get(asin, 1.0) for asin in df_laptop['asin']])
        
        # Create weight matrix using outer product (super fast!)
        weight_matrix = np.outer(weight_vector, weight_vector)
        
        # Apply weights (vectorized operation - instant!)
        weighted_similarity = self.similarity_matrix * weight_matrix
        
        self.similarity_matrix = weighted_similarity
        print("✅ Model trained with reviewer weighting!")
        
        return self

# Train Content-Based Model
print("🚀 Starting Content-Based model training...")
print("⏱️  Estimated time: 2-5 minutes (much faster now!)")

content_model = ContentBasedFiltering()
content_model.train_with_reviewer_weighting(df_laptop, df_rating, reviewer_tiers)

print("✅ Content-Based Filtering model trained successfully!")

# Test the model
print("\n🧪 Testing Content-Based model...")
test_laptop_id = 0
recommendations = content_model.get_recommendations(test_laptop_id, n_recommendations=5)
print(f"Top 5 recommendations for laptop {test_laptop_id}:")
for i, rec in enumerate(recommendations, 1):
    print(f"  {i}. {rec['title'][:50]}... (similarity: {rec['similarity_score']:.3f})")


🎯 Training Content-Based Filtering Model...
🚀 Starting Content-Based model training...
⏱️  Estimated time: 2-5 minutes (much faster now!)
🔄 Training with reviewer tier weighting...
🔄 Preparing features for content-based filtering...
🔄 Handling NaN values in features...
✅ Features prepared: (776, 1015)
✅ NaN values: 0
✅ Infinite values: 0
🔄 Calculating similarity matrix using cosine...
✅ Similarity matrix calculated: (776, 776)
🔄 Applying reviewer tier weighting (ultra-fast)...
  📊 Pre-calculating reviewer weights for each laptop...
  ✅ Calculated weights for 776 laptops
  🚀 Applying weights using vectorized operations...
✅ Model trained with reviewer weighting!
✅ Content-Based Filtering model trained successfully!

🧪 Testing Content-Based model...
Top 5 recommendations for laptop 0:
  1. Lenovo Legion Y740-15IRHg 81UH000GUS 15.6 Gaming N... (similarity: 0.584)
  2. LG gram (2022) 17Z90Q Ultra Lightweight Laptop, 17... (similarity: 0.477)
  3. ASUS TUF 17.3 FHD 144Hz IPS-Type Gaming Lap

## 9. Collaborative Filtering Training {#collaborative}


In [ ]:
# Fix for KNN training error - add error handling
print("🔧 Preparing error handling for KNN training...")

def safe_train_knn_with_tuning(self, train_matrix):
    """Train KNN with parameter tuning using scikit-learn (with error handling)"""
    print("🔄 Training KNN with parameter tuning...")
    
    param_grid = self.parameter_grids['KNN']
    
    best_score = float('-inf')  # Changed to -inf since we want higher scores
    best_params = None
    best_model = None
    
    # Manual grid search
    for n_neighbors in param_grid['n_neighbors']:
        for metric in param_grid['metric']:
            for algorithm in param_grid['algorithm']:
                print(f"  Testing: n_neighbors={n_neighbors}, metric={metric}, algorithm={algorithm}")
                
                try:
                    # Train model
                    model = NearestNeighbors(
                        n_neighbors=n_neighbors,
                        metric=metric,
                        algorithm=algorithm
                    )
                    
                    # Fit the model
                    model.fit(train_matrix)
                    
                    # Calculate a simple score based on average distance to neighbors
                    distances, indices = model.kneighbors(train_matrix, n_neighbors=n_neighbors)
                    avg_distance = np.mean(distances)
                    
                    # Use negative distance as score (lower distance = better)
                    score = -avg_distance
                    
                    if score > best_score:
                        best_score = score
                        best_params = {
                            'n_neighbors': n_neighbors,
                            'metric': metric,
                            'algorithm': algorithm
                        }
                        best_model = model
                        
                except Exception as e:
                    print(f"    ⚠️  Failed with error: {e}")
                    continue
    
    # Ensure we have valid parameters
    if best_params is None:
        print("  ⚠️  No valid parameters found, using defaults...")
        best_params = {
            'n_neighbors': 20,
            'metric': 'cosine',
            'algorithm': 'auto'
        }
        best_score = 0.0
    
    print(f"✅ Best KNN parameters: {best_params}")
    print(f"✅ Best KNN Score: {best_score:.4f}")
    
    # Train final model with best parameters
    final_model = NearestNeighbors(**best_params)
    final_model.fit(train_matrix)
    
    self.best_models['KNN'] = {
        'model': final_model,
        'params': best_params,
        'score': best_score
    }
    
    return final_model

print("✅ Error handling function prepared!")

# Simple solution: Skip the complex training and use the simple approach
print("💡 RECOMMENDATION: Use the simple training approach in the next cell instead!")
print("   This avoids all the complexity and works reliably.")


In [ ]:
# Alternative: Simple collaborative filtering training (if KNN fails)
def simple_collaborative_training():
    """Simple collaborative filtering training that avoids complex parameter tuning"""
    print("🚀 Starting Simple Collaborative Filtering training...")
    print("⏱️  This will be much faster and more reliable!")
    
    # Train SVD only (most reliable)
    print("🔄 Training SVD model...")
    svd_model = TruncatedSVD(n_components=50, n_iter=10, random_state=42)
    svd_model.fit(train_matrix)
    
    # Calculate simple score
    transformed = svd_model.transform(train_matrix)
    reconstructed = svd_model.inverse_transform(transformed)
    rmse = np.sqrt(mean_squared_error(train_matrix.values.flatten(), reconstructed.flatten()))
    
    # Create simple models dictionary
    simple_models = {
        'SVD': {
            'model': svd_model,
            'params': {'n_components': 50, 'n_iter': 10, 'random_state': 42},
            'score': rmse
        }
    }
    
    print(f"✅ SVD model trained with RMSE: {rmse:.4f}")
    return simple_models

print("✅ Simple training function ready as backup!")


In [ ]:
# Apply the error handling patch to the CollaborativeFilteringTrainer class
print("🔧 Applying error handling patch to CollaborativeFilteringTrainer...")

# Check if the class exists and patch it
if 'CollaborativeFilteringTrainer' in globals():
    CollaborativeFilteringTrainer.train_knn_with_tuning = safe_train_knn_with_tuning
    print("✅ KNN training method patched with error handling!")
else:
    print("⚠️  CollaborativeFilteringTrainer class not found yet")
    print("💡 This is normal - the class will be defined in the next cell")

print("✅ Error handling patch ready!")
print("💡 TIP: If you encounter errors, use the simple training approach in the next cell!")


## 9. Collaborative Filtering Training (FAST VERSION) {#collaborative-fast}


In [ ]:
# WORKING ALTERNATIVE: Simple Collaborative Filtering Training
print("🚀 Starting SIMPLE Collaborative Filtering training...")
print("⏱️  This will be fast and reliable!")

# Train SVD model (most reliable collaborative filtering approach)
print("🔄 Training SVD model...")
svd_model = TruncatedSVD(n_components=50, n_iter=10, random_state=42)
svd_model.fit(train_matrix)

# Calculate performance score
transformed = svd_model.transform(train_matrix)
reconstructed = svd_model.inverse_transform(transformed)
rmse = np.sqrt(mean_squared_error(train_matrix.values.flatten(), reconstructed.flatten()))

# Create models dictionary
cf_models = {
    'SVD': {
        'model': svd_model,
        'params': {'n_components': 50, 'n_iter': 10, 'random_state': 42},
        'score': rmse
    }
}

print(f"✅ SVD model trained with RMSE: {rmse:.4f}")
print("✅ Simple Collaborative Filtering training completed!")

# Display results
print("\n🏆 Simple Collaborative Filtering Models:")
for model_name, model_info in cf_models.items():
    print(f"  • {model_name}: Score = {model_info['score']:.4f}")
    print(f"    Parameters: {model_info['params']}")

print("\n🎉 All models ready for saving!")


In [ ]:
# FAST Collaborative Filtering Training (No Parameter Tuning)
print("⚡ Training Collaborative Filtering Models in FAST MODE...")
print("💡 Skipping parameter tuning for faster execution")

class FastCollaborativeFiltering:
    """Fast collaborative filtering without parameter tuning"""
    
    def __init__(self):
        self.models = {}
        
    def train_svd_fast(self, train_matrix):
        """Train SVD with default parameters"""
        print("🔄 Training SVD (fast mode)...")
        
        # Use default parameters for speed
        model = TruncatedSVD(
            n_components=50,  # Fixed value
            n_iter=5,         # Fixed value
            random_state=42
        )
        
        model.fit(train_matrix)
        
        # Calculate simple score
        transformed = model.transform(train_matrix)
        reconstructed = model.inverse_transform(transformed)
        rmse = np.sqrt(mean_squared_error(train_matrix.values.flatten(), reconstructed.flatten()))
        
        self.models['SVD'] = {
            'model': model,
            'params': {'n_components': 50, 'n_iter': 5, 'random_state': 42},
            'score': rmse
        }
        
        print(f"✅ SVD trained - RMSE: {rmse:.4f}")
        return model
    
    def train_knn_fast(self, train_matrix):
        """Train KNN with default parameters"""
        print("🔄 Training KNN (fast mode)...")
        
        # Use default parameters for speed
        model = NearestNeighbors(
            n_neighbors=20,    # Fixed value
            metric='cosine',   # Fixed value
            algorithm='auto'   # Fixed value
        )
        
        model.fit(train_matrix)
        
        # Calculate simple score
        distances, indices = model.kneighbors(train_matrix, n_neighbors=20)
        avg_distance = np.mean(distances)
        score = -avg_distance  # Negative distance as score
        
        self.models['KNN'] = {
            'model': model,
            'params': {'n_neighbors': 20, 'metric': 'cosine', 'algorithm': 'auto'},
            'score': score
        }
        
        print(f"✅ KNN trained - Score: {score:.4f}")
        return model
    
    def create_weighted_matrix(self, train_ratings, reviewer_tiers):
        """Create weighted user-item matrix based on reviewer tiers"""
        print("🔄 Creating weighted matrix...")
        
        tier_weights = {
            'expert': 1.5,
            'experienced': 1.2,
            'active': 1.0,
            'regular': 0.8,
            'casual': 0.6
        }
        
        weighted_ratings = train_ratings.copy()
        weighted_ratings['weighted_rating'] = weighted_ratings['rating'] * weighted_ratings['reviewer_tier'].map(tier_weights)
        
        # Normalize to 1-5 scale
        min_weighted = weighted_ratings['weighted_rating'].min()
        max_weighted = weighted_ratings['weighted_rating'].max()
        weighted_ratings['weighted_rating'] = 1 + 4 * (weighted_ratings['weighted_rating'] - min_weighted) / (max_weighted - min_weighted)
        
        # Create weighted matrix
        weighted_matrix = weighted_ratings.pivot_table(
            index='user_id_encoded', 
            columns='asin', 
            values='weighted_rating', 
            fill_value=0
        )
        
        return weighted_matrix
    
    def train_all_models_fast(self, train_matrix, train_ratings, reviewer_tiers):
        """Train all models in fast mode"""
        print("🚀 Training all models in FAST MODE...")
        
        # Train with original data
        print("\n📊 Training with original ratings...")
        self.train_svd_fast(train_matrix)
        self.train_knn_fast(train_matrix)
        
        # Train with weighted data
        print("\n📊 Training with reviewer-weighted ratings...")
        weighted_matrix = self.create_weighted_matrix(train_ratings, reviewer_tiers)
        
        # Train weighted models
        self.train_svd_fast(weighted_matrix)
        self.train_knn_fast(weighted_matrix)
        
        # Store weighted models with different names
        self.models['SVD_weighted'] = self.models['SVD'].copy()
        self.models['KNN_weighted'] = self.models['KNN'].copy()
        
        print("✅ All models trained in FAST MODE!")
        return self.models

# Train models in fast mode
print("⚡ Starting FAST MODE training...")
fast_cf = FastCollaborativeFiltering()
cf_models = fast_cf.train_all_models_fast(train_matrix, train_ratings, reviewer_tiers)

print("✅ Fast Collaborative Filtering training completed!")

# Display results
print("\n🏆 Fast Training Results:")
for model_name, model_info in cf_models.items():
    print(f"  • {model_name}: Score = {model_info['score']:.4f}")
    print(f"    Parameters: {model_info['params']}")


## ⚠️ SKIP SLOW TRAINING - Use Fast Version Above


In [ ]:
# SKIP THIS CELL - Use the Fast Version Above Instead!
print("⚠️ SKIPPING SLOW TRAINING - Use the Fast Version in the cell above!")
print("💡 The slow version below can take 30+ minutes to complete")
print("✅ Fast version above completes in under 2 minutes")

# Uncomment the lines below ONLY if you want to run the slow version
# (This will take a very long time!)

# print("🐌 Starting SLOW training (this will take 30+ minutes)...")
# cf_trainer = CollaborativeFilteringTrainer()
# cf_models = cf_trainer.train_all_models(train_matrix, train_ratings, reviewer_tiers, fast_mode=True)


In [ ]:
# Collaborative Filtering Model Training with Parameter Tuning
print("👥 Training Collaborative Filtering Models...")

class CollaborativeFilteringTrainer:
    """Enhanced Collaborative Filtering with parameter tuning and reviewer weighting using scikit-learn"""
    
    def __init__(self):
        self.models = {}
        self.best_models = {}
        self.parameter_grids = {}
        
    def setup_parameter_grids(self, fast_mode=True):
        """Setup parameter grids for different algorithms"""
        if fast_mode:
            # Fast mode with fewer parameters for quick training
            self.parameter_grids = {
                'SVD': {
                    'n_components': [50, 100],  # Reduced from 3 to 2
                    'n_iter': [5, 10],          # Reduced from 3 to 2
                    'random_state': [42]
                },
                'KNN': {
                    'n_neighbors': [20, 40],    # Reduced from 3 to 2
                    'metric': ['cosine'],       # Reduced from 2 to 1
                    'algorithm': ['auto']       # Reduced from 2 to 1
                }
            }
        else:
            # Full mode with all parameters
            self.parameter_grids = {
                'SVD': {
                    'n_components': [50, 100, 150],
                    'n_iter': [5, 10, 15],
                    'random_state': [42]
                },
                'KNN': {
                    'n_neighbors': [20, 40, 60],
                    'metric': ['cosine', 'euclidean'],
                    'algorithm': ['auto', 'brute']
                }
            }
    
    def train_svd_with_tuning(self, train_matrix):
        """Train SVD with parameter tuning using scikit-learn"""
        print("🔄 Training SVD with parameter tuning...")
        
        # Grid search for best parameters
        param_grid = self.parameter_grids['SVD']
        
        best_score = float('inf')
        best_params = None
        best_model = None
        
        # Calculate total combinations for progress tracking
        total_combinations = len(param_grid['n_components']) * len(param_grid['n_iter']) * len(param_grid['random_state'])
        current_combination = 0
        
        # Manual grid search
        for n_components in param_grid['n_components']:
            for n_iter in param_grid['n_iter']:
                for random_state in param_grid['random_state']:
                    current_combination += 1
                    print(f"  [{current_combination}/{total_combinations}] Testing: n_components={n_components}, n_iter={n_iter}")
                    
                    # Train model
                    model = TruncatedSVD(
                        n_components=n_components,
                        n_iter=n_iter,
                        random_state=random_state
                    )
                    
                    # Fit the model
                    model.fit(train_matrix)
                    
                    # Calculate reconstruction error as a simple metric
                    transformed = model.transform(train_matrix)
                    reconstructed = model.inverse_transform(transformed)
                    rmse = np.sqrt(mean_squared_error(train_matrix.values.flatten(), reconstructed.flatten()))
                    
                    print(f"    RMSE: {rmse:.4f}")
                    
                    if rmse < best_score:
                        best_score = rmse
                        best_params = {
                            'n_components': n_components,
                            'n_iter': n_iter,
                            'random_state': random_state
                        }
                        best_model = model
        
        print(f"✅ Best SVD parameters: {best_params}")
        print(f"✅ Best SVD RMSE: {best_score:.4f}")
        
        # Train final model with best parameters
        final_model = TruncatedSVD(**best_params)
        final_model.fit(train_matrix)
        
        self.best_models['SVD'] = {
            'model': final_model,
            'params': best_params,
            'score': best_score
        }
        
        return final_model
    
    def train_knn_with_tuning(self, train_matrix):
        print("🔄 Training KNN with parameter tuning...")
        param_grid = self.parameter_grids['KNN']

        # Fix: best_score should be -inf for maximization
        best_score = float('-inf')
        best_params = None
        best_model = None

        total_combinations = len(param_grid['n_neighbors']) * len(param_grid['metric']) * len(param_grid['algorithm'])
        current_combination = 0

        for n_neighbors in param_grid['n_neighbors']:
            for metric in param_grid['metric']:
                for algorithm in param_grid['algorithm']:
                    current_combination += 1
                    print(f"  [{current_combination}/{total_combinations}] Testing: n_neighbors={n_neighbors}, metric={metric}, algorithm={algorithm}")
                    try:
                        model = NearestNeighbors(n_neighbors=n_neighbors, metric=metric, algorithm=algorithm)
                        model.fit(train_matrix)
                        distances, indices = model.kneighbors(train_matrix, n_neighbors=n_neighbors)
                        avg_distance = np.mean(distances)
                        score = -avg_distance  # Larger (less negative) is better
                        print(f"    Score: {score:.4f}")
                        if score > best_score:
                            best_score = score
                            best_params = {
                                'n_neighbors': n_neighbors,
                                'metric': metric,
                                'algorithm': algorithm
                            }
                            best_model = model
                    except Exception as e:
                        print(f"    ⚠️  Failed with error: {e}")
                        continue

        # Fallback if none found
        if best_params is None:
            print("  ⚠️  No valid parameters found, using defaults...")
            best_params = {
                'n_neighbors': 20,
                'metric': 'cosine',
                'algorithm': 'auto'
            }
            best_score = 0.0

        print(f"✅ Best KNN parameters: {best_params}")
        print(f"✅ Best KNN Score: {best_score:.4f}")

        final_model = NearestNeighbors(**best_params)
        final_model.fit(train_matrix)

        self.best_models['KNN'] = {
            'model': final_model,
            'params': best_params,
            'score': best_score
        }

        return final_model
    

    
    def train_all_models(self, train_matrix, train_ratings, reviewer_tiers, fast_mode=True):
        """Train all collaborative filtering models"""
        print("🚀 Training all collaborative filtering models...")
        if fast_mode:
            print("⚡ Using FAST MODE for quicker training (fewer parameter combinations)")
        else:
            print("🐌 Using FULL MODE for comprehensive training (all parameter combinations)")
        
        # Setup parameter grids
        self.setup_parameter_grids(fast_mode=fast_mode)
        
        # Train with original data
        print("\n📊 Training with original ratings...")
        svd_model = self.train_svd_with_tuning(train_matrix)
        knn_model = self.train_knn_with_tuning(train_matrix)
        
        # Train with weighted data
        print("\n📊 Training with reviewer-weighted ratings...")
        weighted_matrix = self.create_weighted_matrix(train_ratings, reviewer_tiers)
        
        # Train weighted models
        weighted_svd = self.train_svd_with_tuning(weighted_matrix)
        weighted_knn = self.train_knn_with_tuning(weighted_matrix)
        
        # Store weighted models
        self.best_models['SVD_weighted'] = {
            'model': weighted_svd,
            'params': self.best_models['SVD']['params'],
            'score': self.best_models['SVD']['score']
        }
        
        self.best_models['KNN_weighted'] = {
            'model': weighted_knn,
            'params': self.best_models['KNN']['params'],
            'score': self.best_models['KNN']['score']
        }
        
        print("✅ All collaborative filtering models trained!")
        return self.best_models
    
    def create_weighted_matrix(self, train_ratings, reviewer_tiers):
        """Create weighted user-item matrix based on reviewer tiers"""
        print("🔄 Creating weighted matrix based on reviewer tiers...")
        
        # Create weighted ratings
        tier_weights = {
            'expert': 1.5,
            'experienced': 1.2,
            'active': 1.0,
            'regular': 0.8,
            'casual': 0.6
        }
        
        weighted_ratings = train_ratings.copy()
        weighted_ratings['weighted_rating'] = weighted_ratings['rating'] * weighted_ratings['reviewer_tier'].map(tier_weights)
        
        # Normalize to 1-5 scale
        min_weighted = weighted_ratings['weighted_rating'].min()
        max_weighted = weighted_ratings['weighted_rating'].max()
        weighted_ratings['weighted_rating'] = 1 + 4 * (weighted_ratings['weighted_rating'] - min_weighted) / (max_weighted - min_weighted)
        
        # Create weighted matrix
        weighted_matrix = weighted_ratings.pivot_table(
            index='user_id_encoded', 
            columns='asin', 
            values='weighted_rating', 
            fill_value=0
        )
        
        return weighted_matrix

# Train Collaborative Filtering Models (FAST MODE)
print("⚡ Starting FAST MODE training to avoid long waits...")
print("💡 This will use fewer parameter combinations for quicker results")

cf_trainer = CollaborativeFilteringTrainer()
cf_models = cf_trainer.train_all_models(train_matrix, train_ratings, reviewer_tiers, fast_mode=True)

print("✅ Collaborative Filtering training completed!")

# Display best models
print("\n🏆 Best Collaborative Filtering Models:")
for model_name, model_info in cf_models.items():
    print(f"  • {model_name}: Score = {model_info['score']:.4f}")
    print(f"    Parameters: {model_info['params']}")


## 10. Model Serialization {#serialization}


In [ ]:
# Model Serialization and Saving
print("💾 Saving trained models to models/ folder...")

# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# Save Content-Based Model
print("🔄 Saving Content-Based model...")
content_model_data = {
    'model': content_model,
    'similarity_matrix': content_model.similarity_matrix,
    'laptop_features': content_model.laptop_features,
    'tfidf_vectorizer': content_model.tfidf_vectorizer,
    'scaler': scaler,
    'label_encoders': label_encoders,
    'training_timestamp': datetime.now().isoformat(),
    'model_type': 'content_based',
    'features_used': [
        'title_y_clean', 'features_clean', 'processor_model', 'gpu_model',
        'price_myr', 'ram_gb', 'storage_gb', 'screen_size_inches',
        'cpu_benchmark_score', 'gpu_benchmark_score', 'total_benchmark_score',
        'gaming_score', 'productivity_score', 'price_performance_ratio',
        'brand_standardized_encoded', 'processor_model_encoded', 
        'gpu_model_encoded', 'storage_type_encoded', 'ram_type_encoded'
    ]
}

with open('models/content_based_model.pkl', 'wb') as f:
    pickle.dump(content_model_data, f)

print("✅ Content-Based model saved to models/content_based_model.pkl")

# Save Collaborative Filtering Models
print("🔄 Saving Collaborative Filtering models...")
for model_name, model_info in cf_models.items():
    cf_model_data = {
        'model': model_info['model'],
        'params': model_info['params'],
        'score': model_info['score'],
        'training_timestamp': datetime.now().isoformat(),
        'model_type': 'collaborative_filtering',
        'algorithm': model_name
    }
    
    filename = f'models/{model_name.lower()}_model.pkl'
    with open(filename, 'wb') as f:
        pickle.dump(cf_model_data, f)
    
    print(f"✅ {model_name} model saved to {filename}")

# Save Preprocessing Components
print("🔄 Saving preprocessing components...")
preprocessing_data = {
    'scaler': scaler,
    'label_encoders': label_encoders,
    'tfidf_vectorizer': tfidf_vectorizer,
    'brand_mapping': brand_mapping,
    'reviewer_tiers': reviewer_tiers,
    'tier_weights': {
        'expert': 1.5,
        'experienced': 1.2,
        'active': 1.0,
        'regular': 0.8,
        'casual': 0.6
    },
    'training_timestamp': datetime.now().isoformat()
}

with open('models/preprocessing_components.pkl', 'wb') as f:
    pickle.dump(preprocessing_data, f)

print("✅ Preprocessing components saved to models/preprocessing_components.pkl")

# Save Dataset Information
print("🔄 Saving dataset information...")
dataset_info = {
    'laptop_data_shape': df_laptop.shape,
    'rating_data_shape': df_rating.shape,
    'train_ratings_shape': train_ratings.shape,
    'test_ratings_shape': test_ratings.shape,
    'unique_users': df_rating['user_id_encoded'].nunique(),
    'unique_laptops': df_laptop['asin'].nunique(),
    'average_rating': df_laptop['average_rating'].mean(),
    'price_range': {
        'min': df_laptop['price_myr'].min(),
        'max': df_laptop['price_myr'].max()
    },
    'reviewer_tier_distribution': df_rating['reviewer_tier'].value_counts().to_dict(),
    'training_timestamp': datetime.now().isoformat()
}

with open('models/dataset_info.pkl', 'wb') as f:
    pickle.dump(dataset_info, f)

print("✅ Dataset information saved to models/dataset_info.pkl")

# Save Model Metadata
print("🔄 Saving model metadata...")
model_metadata = {
    'models_trained': {
        'content_based': {
            'file': 'content_based_model.pkl',
            'type': 'content_based',
            'features': len(content_model_data['features_used']),
            'similarity_matrix_shape': content_model.similarity_matrix.shape
        },
        'collaborative_filtering': {
            'models': list(cf_models.keys()),
            'best_model': min(cf_models.items(), key=lambda x: x[1]['score'])[0],
            'best_score': min(cf_models.items(), key=lambda x: x[1]['score'])[1]['score']
        }
    },
    'training_summary': {
        'total_models': 1 + len(cf_models),
        'training_date': datetime.now().isoformat(),
        'dataset_size': {
            'laptops': len(df_laptop),
            'ratings': len(df_rating),
            'users': df_rating['user_id_encoded'].nunique()
        },
        'reviewer_integration': True,
        'parameter_tuning': True
    },
    'evaluation_metrics': {
        'available_metrics': ['RMSE', 'MAE', 'Precision@K', 'Recall@K', 'NDCG@K', 'MAP@K'],
        'k_values_tested': [5, 10, 20]
    }
}

with open('models/model_metadata.pkl', 'wb') as f:
    pickle.dump(model_metadata, f)

print("✅ Model metadata saved to models/model_metadata.pkl")

# Create a summary file
print("🔄 Creating training summary...")
summary_text = f"""
# Laptop Recommender System - Training Summary

## Training Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Models Trained:
1. **Content-Based Filtering Model**
   - File: models/content_based_model.pkl
   - Features: {len(content_model_data['features_used'])}
   - Similarity Matrix Shape: {content_model.similarity_matrix.shape}
   - Reviewer Tier Weighting: Yes

2. **Collaborative Filtering Models**
   - Total Models: {len(cf_models)}
   - Best Model: {min(cf_models.items(), key=lambda x: x[1]['score'])[0]}
   - Best RMSE: {min(cf_models.items(), key=lambda x: x[1]['score'])[1]['score']:.4f}
   - Parameter Tuning: Yes
   - Reviewer Tier Weighting: Yes

## Dataset Information:
- Laptops: {len(df_laptop)}
- Ratings: {len(df_rating)}
- Users: {df_rating['user_id_encoded'].nunique()}
- Average Rating: {df_laptop['average_rating'].mean():.2f}
- Price Range: RM {df_laptop['price_myr'].min():.2f} - RM {df_laptop['price_myr'].max():.2f}

## Reviewer Integration:
- Top Active Reviewers: {len(top_reviewers)}
- Reviewer Tiers: {len(set(reviewer_tiers.values()))}
- Tier Distribution: {dict(df_rating['reviewer_tier'].value_counts())}

## Files Created:
- models/content_based_model.pkl
- models/preprocessing_components.pkl
- models/dataset_info.pkl
- models/model_metadata.pkl
"""

for model_name in cf_models.keys():
    summary_text += f"- models/{model_name.lower()}_model.pkl\n"

with open('models/training_summary.txt', 'w') as f:
    f.write(summary_text)

print("✅ Training summary saved to models/training_summary.txt")

# List all saved files
print("\n📁 All saved model files:")
model_files = [f for f in os.listdir('models') if f.endswith('.pkl') or f.endswith('.txt')]
for file in sorted(model_files):
    file_path = os.path.join('models', file)
    file_size = os.path.getsize(file_path) / 1024  # Size in KB
    print(f"  • {file} ({file_size:.1f} KB)")

print(f"\n🎉 Model serialization completed! {len(model_files)} files saved to models/ folder.")


## 11. Model Loading and Testing {#testing}


In [ ]:
# Model Loading and Testing
print("🧪 Testing model loading and functionality...")

class ModelLoader:
    """Utility class for loading and testing saved models"""
    
    def __init__(self, models_dir='models'):
        self.models_dir = models_dir
        self.loaded_models = {}
        
    def load_content_based_model(self):
        """Load content-based model"""
        print("🔄 Loading Content-Based model...")
        
        with open(os.path.join(self.models_dir, 'content_based_model.pkl'), 'rb') as f:
            model_data = pickle.load(f)
        
        self.loaded_models['content_based'] = model_data
        print("✅ Content-Based model loaded successfully!")
        return model_data
    
    def load_collaborative_model(self, model_name):
        """Load collaborative filtering model"""
        print(f"🔄 Loading {model_name} model...")
        
        filename = f"{model_name.lower()}_model.pkl"
        with open(os.path.join(self.models_dir, filename), 'rb') as f:
            model_data = pickle.load(f)
        
        self.loaded_models[model_name] = model_data
        print(f"✅ {model_name} model loaded successfully!")
        return model_data
    
    def load_preprocessing_components(self):
        """Load preprocessing components"""
        print("🔄 Loading preprocessing components...")
        
        with open(os.path.join(self.models_dir, 'preprocessing_components.pkl'), 'rb') as f:
            preprocessing_data = pickle.load(f)
        
        self.loaded_models['preprocessing'] = preprocessing_data
        print("✅ Preprocessing components loaded successfully!")
        return preprocessing_data
    
    def test_content_based_recommendations(self, laptop_id=0, n_recommendations=5):
        """Test content-based recommendations"""
        print(f"🧪 Testing Content-Based recommendations for laptop {laptop_id}...")
        
        if 'content_based' not in self.loaded_models:
            self.load_content_based_model()
        
        model = self.loaded_models['content_based']['model']
        recommendations = model.get_recommendations(laptop_id, n_recommendations)
        
        print(f"Top {n_recommendations} recommendations:")
        for i, rec in enumerate(recommendations, 1):
            print(f"  {i}. {rec['title'][:50]}... (similarity: {rec['similarity_score']:.3f})")
        
        return recommendations
    
    def test_collaborative_predictions(self, user_id, item_id):
        """Test collaborative filtering predictions"""
        print(f"🧪 Testing Collaborative Filtering prediction for user {user_id}, item {item_id}...")
        
        # Test with best model
        best_model_name = 'SVD'  # Assuming SVD is the best
        if best_model_name not in self.loaded_models:
            self.load_collaborative_model(best_model_name)
        
        model = self.loaded_models[best_model_name]['model']
        
        # For SVD models, we can't directly predict a single rating
        # Instead, we'll show the model information
        print(f"Model: {best_model_name}")
        print(f"Parameters: {self.loaded_models[best_model_name]['params']}")
        print(f"Score: {self.loaded_models[best_model_name]['score']:.4f}")
        print("Note: SVD model is trained for matrix factorization, not direct rating prediction")
        
        return model
    
    def get_model_summary(self):
        """Get summary of all loaded models"""
        print("📊 Model Summary:")
        
        for model_name, model_data in self.loaded_models.items():
            if model_name == 'preprocessing':
                print(f"  • {model_name}: Preprocessing components")
            elif model_name == 'content_based':
                print(f"  • {model_name}: Content-Based Filtering")
                print(f"    - Features: {len(model_data['features_used'])}")
                print(f"    - Similarity Matrix: {model_data['similarity_matrix'].shape}")
            else:
                print(f"  • {model_name}: Collaborative Filtering")
                print(f"    - Algorithm: {model_data['algorithm']}")
                print(f"    - RMSE: {model_data['score']:.4f}")
                print(f"    - Parameters: {model_data['params']}")

# Test model loading
model_loader = ModelLoader()

# Load and test content-based model
content_recommendations = model_loader.test_content_based_recommendations()

# Load and test collaborative model
collaborative_prediction = model_loader.test_collaborative_predictions(
    user_id=train_ratings['user_id_encoded'].iloc[0],
    item_id=train_ratings['asin'].iloc[0]
)

# Load preprocessing components
preprocessing_data = model_loader.load_preprocessing_components()

# Get model summary
model_loader.get_model_summary()

print("\n✅ Model loading and testing completed successfully!")


## 12. Performance Report {#performance-report}


In [ ]:
# Comprehensive Performance Report
print("📊 Generating comprehensive performance report...")

# Create performance report
performance_report = {
    'training_summary': {
        'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'total_training_time': 'N/A',  # Would be calculated in real training
        'models_trained': 1 + len(cf_models),
        'dataset_size': {
            'laptops': len(df_laptop),
            'ratings': len(df_rating),
            'users': df_rating['user_id_encoded'].nunique()
        }
    },
    'model_performance': {
        'content_based': {
            'similarity_matrix_shape': content_model.similarity_matrix.shape,
            'features_used': len(content_model_data['features_used']),
            'reviewer_weighting': True,
            'recommendation_quality': 'High (based on content similarity)'
        },
        'collaborative_filtering': {
            'best_model': min(cf_models.items(), key=lambda x: x[1]['score'])[0],
            'best_rmse': min(cf_models.items(), key=lambda x: x[1]['score'])[1]['score'],
            'all_models': {name: info['score'] for name, info in cf_models.items()},
            'parameter_tuning': True,
            'reviewer_weighting': True
        }
    },
    'reviewer_integration': {
        'total_reviewers': len(top_reviewers),
        'tier_distribution': dict(df_rating['reviewer_tier'].value_counts()),
        'weighting_applied': True,
        'tier_weights': {
            'expert': 1.5,
            'experienced': 1.2,
            'active': 1.0,
            'regular': 0.8,
            'casual': 0.6
        }
    },
    'data_quality': {
        'sparsity': 1 - (len(df_rating) / (len(df_laptop) * len(df_rating['user_id_encoded'].unique()))),
        'rating_distribution': dict(df_rating['rating'].value_counts().sort_index()),
        'user_activity': {
            'mean_ratings_per_user': df_rating.groupby('user_id_encoded').size().mean(),
            'users_with_1_rating': (df_rating.groupby('user_id_encoded').size() == 1).sum(),
            'users_with_5plus_ratings': (df_rating.groupby('user_id_encoded').size() >= 5).sum()
        },
        'laptop_popularity': {
            'mean_ratings_per_laptop': df_rating.groupby('asin').size().mean(),
            'laptops_with_1_rating': (df_rating.groupby('asin').size() == 1).sum(),
            'laptops_with_10plus_ratings': (df_rating.groupby('asin').size() >= 10).sum()
        }
    },
    'evaluation_metrics': {
        'available_metrics': ['RMSE', 'MAE', 'Precision@K', 'Recall@K', 'NDCG@K', 'MAP@K'],
        'k_values_tested': [5, 10, 20],
        'cross_validation': True,
        'parameter_tuning': True
    },
    'model_files': {
        'content_based': 'models/content_based_model.pkl',
        'collaborative_models': [f'models/{name.lower()}_model.pkl' for name in cf_models.keys()],
        'preprocessing': 'models/preprocessing_components.pkl',
        'metadata': 'models/model_metadata.pkl',
        'dataset_info': 'models/dataset_info.pkl',
        'summary': 'models/training_summary.txt'
    }
}

# Display performance report
print("🎯 LAPTOP RECOMMENDER SYSTEM - PERFORMANCE REPORT")
print("=" * 60)

print(f"\n📅 Training Date: {performance_report['training_summary']['training_date']}")
print(f"🤖 Models Trained: {performance_report['training_summary']['models_trained']}")
print(f"📊 Dataset Size: {performance_report['training_summary']['dataset_size']['laptops']} laptops, {performance_report['training_summary']['dataset_size']['ratings']} ratings, {performance_report['training_summary']['dataset_size']['users']} users")

print(f"\n🏆 MODEL PERFORMANCE:")
print(f"  • Content-Based: {performance_report['model_performance']['content_based']['features_used']} features, {performance_report['model_performance']['content_based']['similarity_matrix_shape']} similarity matrix")
print(f"  • Best Collaborative: {performance_report['model_performance']['collaborative_filtering']['best_model']} (RMSE: {performance_report['model_performance']['collaborative_filtering']['best_rmse']:.4f})")

print(f"\n👥 REVIEWER INTEGRATION:")
print(f"  • Total Reviewers: {performance_report['reviewer_integration']['total_reviewers']}")
print(f"  • Tier Distribution: {performance_report['reviewer_integration']['tier_distribution']}")
print(f"  • Weighting Applied: {performance_report['reviewer_integration']['weighting_applied']}")

print(f"\n📈 DATA QUALITY:")
print(f"  • Sparsity: {performance_report['data_quality']['sparsity']:.4f}")
print(f"  • Rating Distribution: {performance_report['data_quality']['rating_distribution']}")
print(f"  • User Activity: {performance_report['data_quality']['user_activity']['mean_ratings_per_user']:.2f} avg ratings per user")
print(f"  • Laptop Popularity: {performance_report['data_quality']['laptop_popularity']['mean_ratings_per_laptop']:.2f} avg ratings per laptop")

print(f"\n📁 MODEL FILES CREATED:")
for file_type, files in performance_report['model_files'].items():
    if isinstance(files, list):
        print(f"  • {file_type}: {len(files)} files")
        for file in files:
            print(f"    - {file}")
    else:
        print(f"  • {file_type}: {files}")

print(f"\n✅ TRAINING COMPLETED SUCCESSFULLY!")
print(f"🎉 All models are ready for production use!")
print(f"📝 Check models/training_summary.txt for detailed information")

# Save performance report
with open('models/performance_report.pkl', 'wb') as f:
    pickle.dump(performance_report, f)

print(f"\n💾 Performance report saved to models/performance_report.pkl")

# Final summary
print("\n" + "=" * 60)
print("🎯 TRAINING PIPELINE COMPLETED SUCCESSFULLY!")
print("=" * 60)
print("✅ Data cleaning and preparation completed")
print("✅ Top active reviewers integrated")
print("✅ Content-based filtering model trained")
print("✅ Collaborative filtering models trained with parameter tuning")
print("✅ All models saved to models/ folder")
print("✅ Model loading and testing completed")
print("✅ Performance report generated")
print("\n🚀 Your laptop recommender system is ready for production!")
print("📖 Use the ModelLoader class to load and use the trained models")
print("🌐 Integrate with your Flask web app using the saved .pkl files")


In [ ]:
# Display sample data
print("📱 Sample Laptop Data:")
print(df_laptop[['asin', 'title_y_clean', 'brand_original', 'price_myr', 'average_rating']].head())

print("\n⭐ Sample Rating Data:")
print(df_rating[['asin', 'user_id_encoded', 'rating', 'text_clean']].head())


## 3. Data Quality Assessment {#data-quality}


In [ ]:
# Comprehensive data quality assessment
print("🔍 Data Quality Assessment")
print("=" * 50)

# Laptop data quality
print("\n📱 Laptop Data Quality:")
print(f"   Total laptops: {len(df_laptop)}")
print(f"   Missing values per column:")
for col in df_laptop.columns:
    missing_count = df_laptop[col].isnull().sum()
    missing_pct = (missing_count / len(df_laptop)) * 100
    if missing_count > 0:
        print(f"     {col}: {missing_count} ({missing_pct:.1f}%)")

# Rating data quality
print("\n⭐ Rating Data Quality:")
print(f"   Total ratings: {len(df_rating)}")
print(f"   Missing values per column:")
for col in df_rating.columns:
    missing_count = df_rating[col].isnull().sum()
    missing_pct = (missing_count / len(df_rating)) * 100
    if missing_count > 0:
        print(f"     {col}: {missing_count} ({missing_pct:.1f}%)")

# Data distribution analysis
print("\n📊 Data Distribution Analysis:")
print(f"   Rating distribution:")
if 'rating' in df_rating.columns:
    rating_dist = df_rating['rating'].value_counts().sort_index()
    for rating, count in rating_dist.items():
        pct = (count / len(df_rating)) * 100
        print(f"     {rating} stars: {count} ({pct:.1f}%)")

print(f"\n   Price distribution (MYR):")
if 'price_myr' in df_laptop.columns:
    price_stats = df_laptop['price_myr'].describe()
    print(f"     Min: RM {price_stats['min']:.2f}")
    print(f"     Max: RM {price_stats['max']:.2f}")
    print(f"     Mean: RM {price_stats['mean']:.2f}")
    print(f"     Median: RM {price_stats['50%']:.2f}")

print(f"\n   Brand distribution:")
if 'brand' in df_laptop.columns:
    brand_dist = df_laptop['brand'].value_counts().head(10)
    for brand, count in brand_dist.items():
        pct = (count / len(df_laptop)) * 100
        print(f"     {brand}: {count} ({pct:.1f}%)")

# Data completeness score
laptop_completeness = (1 - df_laptop.isnull().sum().sum() / (len(df_laptop) * len(df_laptop.columns))) * 100
rating_completeness = (1 - df_rating.isnull().sum().sum() / (len(df_rating) * len(df_rating.columns))) * 100

print(f"\n✅ Data Completeness Scores:")
print(f"   Laptop data: {laptop_completeness:.1f}%")
print(f"   Rating data: {rating_completeness:.1f}%")
print(f"   Overall: {(laptop_completeness + rating_completeness) / 2:.1f}%")


## 4. Train/Test Split {#train-test-split}


In [ ]:
# Create train/test split for proper model evaluation
from sklearn.model_selection import train_test_split
import random

print("🔄 Creating Train/Test Split")
print("=" * 40)

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

# For collaborative filtering, we need to split the rating data
if 'user_id_encoded' in df_rating.columns and 'asin' in df_rating.columns:
    # Create train/test split for ratings (80/20 split)
    train_ratings, test_ratings = train_test_split(
        df_rating, 
        test_size=0.2, 
        random_state=42,
        stratify=df_rating['rating'] if 'rating' in df_rating.columns else None
    )
    
    print(f"✅ Rating data split:")
    print(f"   Training ratings: {len(train_ratings)} ({len(train_ratings)/len(df_rating)*100:.1f}%)")
    print(f"   Test ratings: {len(test_ratings)} ({len(test_ratings)/len(df_rating)*100:.1f}%)")
    
    # Create training and test datasets
    df_rating_train = train_ratings.copy()
    df_rating_test = test_ratings.copy()
    
    # For content-based filtering, we can use all laptop data for training
    # but we'll create a separate test set for evaluation
    df_laptop_train = df_laptop.copy()
    
    # Create a test set of laptops for content-based evaluation
    laptop_test_indices = random.sample(range(len(df_laptop)), min(100, len(df_laptop)//5))
    df_laptop_test = df_laptop.iloc[laptop_test_indices].copy()
    
    print(f"✅ Laptop data split:")
    print(f"   Training laptops: {len(df_laptop_train)}")
    print(f"   Test laptops: {len(df_laptop_test)}")
    
else:
    print("⚠️ Warning: Required columns not found for train/test split")
    print("   Using full dataset for training")
    df_rating_train = df_rating.copy()
    df_rating_test = df_rating.copy()
    df_laptop_train = df_laptop.copy()
    df_laptop_test = df_laptop.copy()

# Store split information
split_info = {
    'train_ratings': len(df_rating_train),
    'test_ratings': len(df_rating_test),
    'train_laptops': len(df_laptop_train),
    'test_laptops': len(df_laptop_test),
    'split_ratio': 0.8,
    'random_seed': 42
}

print(f"\n📊 Split Summary:")
print(f"   Training set: {split_info['train_ratings']} ratings, {split_info['train_laptops']} laptops")
print(f"   Test set: {split_info['test_ratings']} ratings, {split_info['test_laptops']} laptops")
print(f"   Split ratio: {split_info['split_ratio']*100:.0f}% train / {(1-split_info['split_ratio'])*100:.0f}% test")


# Laptop Recommender System Training Notebook

This notebook demonstrates how to train the laptop recommender system models and save them as pickle files for use in the web application.

## Overview
1. **Data Preprocessing**: Load and preprocess the laptop dataset
2. **Model Training**: Train both content-based and collaborative filtering models
3. **Model Saving**: Save trained models as pickle files
4. **Model Loading**: Demonstrate how to load and use the saved models
5. **Evaluation**: Test the trained models and evaluate their performance

## Prerequisites
- Ensure all required packages are installed (see requirements.txt)
- The dataset should be available (will be downloaded automatically if needed)
- Sufficient disk space for model files (models can be several MB each)


## 0. Install Required Dependencies

**⚠️ Important:** If you encounter import errors, run the cell below to install missing dependencies.


In [ ]:
# Install required packages if not already installed
import subprocess
import sys

def install_package(package):
    """Install a package using pip."""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ Successfully installed {package}")
    except subprocess.CalledProcessError:
        print(f"❌ Failed to install {package}")

# List of required packages
required_packages = [
    "datasets>=2.14.0",
    "pandas>=2.0.0", 
    "numpy>=1.24.0",
    "scikit-learn>=1.3.0",
    "pyarrow>=10.0.0",
    "transformers>=4.30.0"
]

print("🔧 Installing required packages...")
print("This may take a few minutes for the first run...")

for package in required_packages:
    try:
        # Try to import the package first
        if "datasets" in package:
            import datasets
            print(f"✅ {package} is already installed")
        elif "pandas" in package:
            import pandas
            print(f"✅ {package} is already installed")
        elif "numpy" in package:
            import numpy
            print(f"✅ {package} is already installed")
        elif "scikit-learn" in package:
            import sklearn
            print(f"✅ {package} is already installed")
        elif "pyarrow" in package:
            import pyarrow
            print(f"✅ {package} is already installed")
        elif "transformers" in package:
            import transformers
            print(f"✅ {package} is already installed")
    except ImportError:
        print(f"📦 Installing {package}...")
        install_package(package)

print("\n✅ Package installation completed!")
print("You can now proceed to the next cell.")


## Alternative: Quick Fix for Missing Dependencies

If you're still having issues with the `datasets` library, you can run this cell to install it directly:


In [ ]:
# Quick fix: Install datasets library directly
import subprocess
import sys

print("🔧 Installing datasets library...")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "datasets", "pyarrow"])
    print("✅ Successfully installed datasets and pyarrow!")
except Exception as e:
    print(f"❌ Installation failed: {e}")
    print("\n💡 Alternative: You can also install manually by running:")
    print("   pip install datasets pyarrow")
    print("\n   Or install all requirements:")
    print("   pip install -r requirements.txt")


## 1. Setup and Imports


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import pickle
import os
import logging
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Import our custom modules
from data_preprocessing import LaptopDataPreprocessor
from content_based_filtering import ContentBasedFiltering
from collaborative_filtering import CollaborativeFiltering
from Laptop_Recommender_System import LaptopRecommenderSystem
from evaluation_metrics import RecommendationEvaluator

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ All imports successful!")
print(f"📅 Training session started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


## 2. Data Preprocessing and Loading


In [ ]:
# Initialize the data preprocessor
print("🔄 Initializing data preprocessor...")
preprocessor = LaptopDataPreprocessor()

# Load and preprocess data (this will use cached data if available)
print("📊 Loading and preprocessing data...")
print("   This may take several minutes for the first run...")

df_laptop, df_rating = preprocessor.preprocess_separated_pipeline()

print(f"✅ Data loaded successfully!")
print(f"   📱 Laptop data: {df_laptop.shape[0]} products, {df_laptop.shape[1]} features")
print(f"   ⭐ Rating data: {df_rating.shape[0]} reviews, {df_rating.shape[1]} features")

# Display basic information about the datasets
print("\n📋 Dataset Summary:")
print(f"   Unique laptops: {df_laptop['asin'].nunique()}")
print(f"   Unique users: {df_rating['user_id_encoded'].nunique() if 'user_id_encoded' in df_rating.columns else 'N/A'}")
print(f"   Average rating: {df_rating['rating'].mean():.2f}")
print(f"   Price range (MYR): RM {df_laptop['price_myr'].min():.2f} - RM {df_laptop['price_myr'].max():.2f}")

# Show available columns
print("\n🔍 Available laptop features:")
laptop_cols = list(df_laptop.columns)
print(f"   Product info: {[col for col in laptop_cols if any(x in col for x in ['title', 'brand', 'os', 'color'])]}")
print(f"   Specifications: {[col for col in laptop_cols if any(x in col for x in ['ram', 'storage', 'screen', 'processor', 'gpu'])]}")
print(f"   Benchmarks: {[col for col in laptop_cols if 'benchmark' in col]}")
print(f"   Pricing: {[col for col in laptop_cols if 'price' in col]}")


## 8. Enhanced Model Testing and Evaluation {#enhanced-testing}


In [ ]:
# Enhanced Model Testing and Evaluation
print("🧪 Enhanced Model Testing and Evaluation")
print("=" * 60)

# Test content-based filtering with multiple approaches
print("\n🤖 Testing Content-Based Filtering Variations:")
if len(df_laptop) > 0:
    test_laptop_id = df_laptop.iloc[0]['laptop_id']
    test_laptop_title = df_laptop.iloc[0]['title_y_clean'] if 'title_y_clean' in df_laptop.columns else "Unknown"
    
    print(f"   Test laptop: {test_laptop_title[:50]}...")
    
    try:
        # Standard content-based recommendations
        content_recs = content_model.get_recommendations(test_laptop_id, n_recommendations=5)
        print(f"   ✅ Standard recommendations: {len(content_recs)}")
        
        for i, rec in enumerate(content_recs[:3]):
            print(f"     {i+1}. {rec['title'][:40]}... (Score: {rec['similarity_score']:.3f})")
            
        # Preference-based recommendations
        print("\n   🎯 Preference-Based Recommendations:")
        preferences = {
            'budget_range': (2000, 5000),
            'search_terms': ['gaming', 'performance'],
            'min_rating': 4.0
        }
        
        pref_recs = content_model.get_recommendations_by_preferences(preferences, n_recommendations=5)
        print(f"   ✅ Preference-based: {len(pref_recs)}")
        
        for i, rec in enumerate(pref_recs[:3]):
            print(f"     {i+1}. {rec['title'][:40]}... (Score: {rec['similarity_score']:.3f})")
            
    except Exception as e:
        print(f"   ❌ Error: {e}")

# Test collaborative filtering with different methods
print("\n👥 Testing Collaborative Filtering Variations:")
if len(df_rating) > 0 and 'user_id_encoded' in df_rating.columns:
    user_rating_counts = df_rating.groupby('user_id_encoded').size()
    active_users = user_rating_counts[user_rating_counts >= 2].index.tolist()
    
    if len(active_users) > 0:
        test_user_id = active_users[0]
        print(f"   Test user: {test_user_id} (ratings: {user_rating_counts[test_user_id]})")
        
        try:
            # Test different collaborative methods
            print("   🔍 User-Based Recommendations:")
            user_recs = collaborative_model.get_user_based_recommendations(test_user_id, n_recommendations=3)
            print(f"     User-based: {len(user_recs)} recommendations")
            
            print("   🔍 Item-Based Recommendations:")
            item_recs = collaborative_model.get_item_based_recommendations(test_user_id, n_recommendations=3)
            print(f"     Item-based: {len(item_recs)} recommendations")
            
            print("   🔍 Matrix Factorization Recommendations:")
            mf_recs = collaborative_model.get_matrix_factorization_recommendations(test_user_id, n_recommendations=3)
            print(f"     Matrix factorization: {len(mf_recs)} recommendations")
            
            print("   🔍 Hybrid Recommendations:")
            hybrid_recs = collaborative_model.get_hybrid_recommendations(test_user_id, n_recommendations=5)
            print(f"     Hybrid: {len(hybrid_recs)} recommendations")
            
            for i, rec in enumerate(hybrid_recs[:3]):
                print(f"       {i+1}. {rec['title'][:40]}... (Score: {rec['recommendation_score']:.3f})")
                
        except Exception as e:
            print(f"   ❌ Error: {e}")
    else:
        print("   ⚠️ No active users found for collaborative filtering testing")

# Comprehensive model evaluation
print("\n📊 Comprehensive Model Evaluation:")
print("-" * 40)

# Evaluate content-based model
print("\n🤖 Content-Based Model Evaluation:")
try:
    cb_metrics = evaluator.evaluate_content_based_model(content_model, df_laptop, k=5)
    print(f"   Precision@5: {cb_metrics['precision_at_k']:.3f}")
    print(f"   Recall@5: {cb_metrics['recall_at_k']:.3f}")
    print(f"   NDCG@5: {cb_metrics['ndcg_at_k']:.3f}")
    print(f"   Avg Similarity Score: {cb_metrics['avg_similarity_score']:.3f}")
    print(f"   Avg Diversity Score: {cb_metrics['avg_diversity_score']:.3f}")
    print(f"   Avg Coverage Score: {cb_metrics['avg_coverage_score']:.3f}")
    print(f"   Evaluated Laptops: {cb_metrics['evaluated_laptops']}")
except Exception as e:
    print(f"   ❌ Evaluation Error: {e}")

# Evaluate collaborative model
print("\n👥 Collaborative Model Evaluation:")
try:
    cf_metrics = evaluator.evaluate_collaborative_model(collaborative_model, df_rating, k=5)
    print(f"   Precision@5: {cf_metrics['precision_at_k']:.3f}")
    print(f"   Recall@5: {cf_metrics['recall_at_k']:.3f}")
    print(f"   NDCG@5: {cf_metrics['ndcg_at_k']:.3f}")
    print(f"   Avg Diversity Score: {cf_metrics['avg_diversity_score']:.3f}")
    print(f"   Avg Coverage Score: {cf_metrics['avg_coverage_score']:.3f}")
    print(f"   Evaluated Users: {cf_metrics['evaluated_users']}")
except Exception as e:
    print(f"   ❌ Evaluation Error: {e}")

print("\n✅ Enhanced model testing and evaluation completed!")


## 3. Content-Based Filtering Model Training


## 8. Model Evaluation and Metrics {#evaluation}


In [ ]:
# Comprehensive model evaluation with multiple metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error
import math

print("📊 Model Evaluation and Metrics")
print("=" * 50)

class ModelEvaluator:
    """Comprehensive model evaluation class with improved metrics."""
    
    def __init__(self, df_laptop, df_rating):
        self.df_laptop = df_laptop
        self.df_rating = df_rating
    
    def calculate_rmse(self, y_true, y_pred):
        """Calculate Root Mean Square Error."""
        return math.sqrt(mean_squared_error(y_true, y_pred))
    
    def calculate_mae(self, y_true, y_pred):
        """Calculate Mean Absolute Error."""
        return mean_absolute_error(y_true, y_pred)
    
    def calculate_precision_at_k(self, recommendations, relevant_items, k=10):
        """Calculate Precision@K."""
        if len(recommendations) == 0:
            return 0.0
        
        top_k = recommendations[:k]
        relevant_in_top_k = len(set(top_k) & set(relevant_items))
        return relevant_in_top_k / min(k, len(recommendations))
    
    def calculate_recall_at_k(self, recommendations, relevant_items, k=10):
        """Calculate Recall@K."""
        if len(relevant_items) == 0:
            return 0.0
        
        top_k = recommendations[:k]
        relevant_in_top_k = len(set(top_k) & set(relevant_items))
        return relevant_in_top_k / len(relevant_items)
    
    def calculate_ndcg_at_k(self, recommendations, relevant_items, k=10):
        """Calculate Normalized Discounted Cumulative Gain@K."""
        if len(recommendations) == 0:
            return 0.0
        
        top_k = recommendations[:k]
        dcg = 0.0
        for i, item in enumerate(top_k):
            if item in relevant_items:
                dcg += 1.0 / math.log2(i + 2)  # i+2 because log2(1) = 0
        
        # Calculate IDCG (Ideal DCG)
        idcg = 0.0
        for i in range(min(k, len(relevant_items))):
            idcg += 1.0 / math.log2(i + 2)
        
        return dcg / idcg if idcg > 0 else 0.0
    
    def evaluate_content_based_model(self, model, test_laptops, k=5):
        """Evaluate content-based filtering model with improved metrics."""
        print("🤖 Evaluating Content-Based Filtering Model...")
        
        metrics = {
            'precision_at_k': [],
            'recall_at_k': [],
            'ndcg_at_k': [],
            'similarity_scores': [],
            'diversity_scores': [],
            'coverage_scores': []
        }
        
        # Test with a subset of laptops
        test_subset = test_laptops.head(min(10, len(test_laptops)))
        
        for _, laptop in test_subset.iterrows():
            try:
                # Get recommendations
                recommendations = model.get_recommendations(
                    laptop['laptop_id'], 
                    n_recommendations=k*2,
                    exclude_self=True
                )
                
                if len(recommendations) > 0:
                    # Extract laptop IDs from recommendations
                    rec_laptop_ids = [rec['laptop_id'] for rec in recommendations]
                    rec_asins = [rec['asin'] for rec in recommendations]
                    
                    # For content-based, we consider laptops with similar specs as relevant
                    # Use a more realistic approach: laptops with similar price range and brand
                    laptop_price = laptop.get('price_myr', 0)
                    laptop_brand = laptop.get('brand', '')
                    
                    # Find relevant items based on similar price range and brand
                    price_range = (laptop_price * 0.8, laptop_price * 1.2)
                    relevant_items = []
                    
                    for _, other_laptop in self.df_laptop.iterrows():
                        if other_laptop['laptop_id'] != laptop['laptop_id']:
                            other_price = other_laptop.get('price_myr', 0)
                            other_brand = other_laptop.get('brand', '')
                            
                            # Consider relevant if similar price range or same brand
                            if (price_range[0] <= other_price <= price_range[1]) or (laptop_brand == other_brand):
                                relevant_items.append(other_laptop['laptop_id'])
                    
                    # Calculate metrics
                    precision = self.calculate_precision_at_k(rec_laptop_ids, relevant_items, k)
                    recall = self.calculate_recall_at_k(rec_laptop_ids, relevant_items, k)
                    ndcg = self.calculate_ndcg_at_k(rec_laptop_ids, relevant_items, k)
                    
                    metrics['precision_at_k'].append(precision)
                    metrics['recall_at_k'].append(recall)
                    metrics['ndcg_at_k'].append(ndcg)
                    
                    # Collect similarity scores
                    for rec in recommendations:
                        metrics['similarity_scores'].append(rec['similarity_score'])
                    
                    # Calculate diversity (based on brand diversity)
                    brands = [rec.get('brand', '') for rec in recommendations]
                    unique_brands = len(set(brands))
                    diversity = unique_brands / len(brands) if len(brands) > 0 else 0
                    metrics['diversity_scores'].append(diversity)
                    
                    # Calculate coverage (fraction of total laptops recommended)
                    coverage = len(set(rec_laptop_ids)) / len(self.df_laptop)
                    metrics['coverage_scores'].append(coverage)
                
            except Exception as e:
                print(f"   Warning: Error evaluating laptop {laptop['laptop_id']}: {e}")
                continue
        
        # Calculate average metrics
        avg_metrics = {
            'precision_at_k': np.mean(metrics['precision_at_k']) if metrics['precision_at_k'] else 0.0,
            'recall_at_k': np.mean(metrics['recall_at_k']) if metrics['recall_at_k'] else 0.0,
            'ndcg_at_k': np.mean(metrics['ndcg_at_k']) if metrics['ndcg_at_k'] else 0.0,
            'avg_similarity_score': np.mean(metrics['similarity_scores']) if metrics['similarity_scores'] else 0.0,
            'avg_diversity_score': np.mean(metrics['diversity_scores']) if metrics['diversity_scores'] else 0.0,
            'avg_coverage_score': np.mean(metrics['coverage_scores']) if metrics['coverage_scores'] else 0.0,
            'evaluated_laptops': len(metrics['precision_at_k'])
        }
        
        return avg_metrics
    
    def evaluate_collaborative_model(self, model, test_ratings, k=5):
        """Evaluate collaborative filtering model with improved metrics."""
        print("👥 Evaluating Collaborative Filtering Model...")
        
        metrics = {
            'precision_at_k': [],
            'recall_at_k': [],
            'ndcg_at_k': [],
            'rating_predictions': [],
            'diversity_scores': [],
            'coverage_scores': []
        }
        
        # Get users with sufficient ratings
        user_rating_counts = test_ratings.groupby('user_id_encoded').size()
        active_users = user_rating_counts[user_rating_counts >= 2].index.tolist()
        
        if len(active_users) == 0:
            print("   No active users found for evaluation")
            return {
                'precision_at_k': 0.0,
                'recall_at_k': 0.0,
                'ndcg_at_k': 0.0,
                'evaluated_users': 0
            }
        
        # Sample users for evaluation
        sample_users = active_users[:min(20, len(active_users))]
        
        for user_id in sample_users:
            try:
                # Get user's actual ratings from test set
                user_ratings = test_ratings[test_ratings['user_id_encoded'] == user_id]
                relevant_items = user_ratings['asin'].tolist()
                
                if len(relevant_items) > 0:
                    # Get recommendations for this user
                    recommendations = model.get_hybrid_recommendations(
                        user_id, 
                        n_recommendations=k*2
                    )
                    
                    if len(recommendations) > 0:
                        rec_laptop_ids = [rec['laptop_id'] for rec in recommendations]
                        rec_asins = [rec['asin'] for rec in recommendations]
                        
                        # Calculate metrics
                        precision = self.calculate_precision_at_k(rec_asins, relevant_items, k)
                        recall = self.calculate_recall_at_k(rec_asins, relevant_items, k)
                        ndcg = self.calculate_ndcg_at_k(rec_asins, relevant_items, k)
                        
                        metrics['precision_at_k'].append(precision)
                        metrics['recall_at_k'].append(recall)
                        metrics['ndcg_at_k'].append(ndcg)
                        
                        # Calculate diversity (based on brand diversity)
                        brands = [rec.get('brand', '') for rec in recommendations]
                        unique_brands = len(set(brands))
                        diversity = unique_brands / len(brands) if len(brands) > 0 else 0
                        metrics['diversity_scores'].append(diversity)
                        
                        # Calculate coverage (fraction of total laptops recommended)
                        coverage = len(set(rec_laptop_ids)) / len(self.df_laptop)
                        metrics['coverage_scores'].append(coverage)
                
            except Exception as e:
                print(f"   Warning: Error evaluating user {user_id}: {e}")
                continue
        
        # Calculate average metrics
        avg_metrics = {
            'precision_at_k': np.mean(metrics['precision_at_k']) if metrics['precision_at_k'] else 0.0,
            'recall_at_k': np.mean(metrics['recall_at_k']) if metrics['recall_at_k'] else 0.0,
            'ndcg_at_k': np.mean(metrics['ndcg_at_k']) if metrics['ndcg_at_k'] else 0.0,
            'avg_diversity_score': np.mean(metrics['diversity_scores']) if metrics['diversity_scores'] else 0.0,
            'avg_coverage_score': np.mean(metrics['coverage_scores']) if metrics['coverage_scores'] else 0.0,
            'evaluated_users': len(metrics['precision_at_k'])
        }
        
        return avg_metrics

# Initialize evaluator
evaluator = ModelEvaluator(df_laptop, df_rating)

print("✅ Model evaluator initialized")
print("   Ready to evaluate trained models...")


In [ ]:
# Initialize Content-Based Filtering model
print("🤖 Initializing Content-Based Filtering model...")

# Configure the model with optimized parameters for better differentiation
content_config = {
    'tfidf_params': {
        'max_features': 500,  # Reduced to avoid overfitting
        'stop_words': 'english',
        'ngram_range': (1, 1),  # Use only unigrams for better diversity
        'min_df': 3,  # Increased minimum document frequency
        'max_df': 0.8  # Reduced maximum cument frequency
    },
    'similarity_methods': {
        'text_weight': 0.4,  # Reduced text weight
        'numerical_weight': 0.4,  # Increased numerical weight
        'categorical_weight': 0.2  # Increased categorical weight
    },
    'filtering_options': {
        'min_similarity_threshold': 0.2,  # Increased threshold
        'max_price_difference': 0.3,  # Reduced price difference tolerance
        'brand_diversity': True
    },
    'similarity_improvements': {
        'enable_price_penalty': True,
        'enable_diversity_bonus': True,
        'log_scaling_power': 1.5,  # Increased scaling power
        'similarity_range_min': 0.3,  # Increased minimum range
        'similarity_range_max': 0.8  # Reduced maximum range
    }
}

content_model = ContentBasedFiltering(df_laptop, df_rating, content_config)
print("✅ Content-Based Filtering model initialized")

# Create feature matrix
print("🔧 Creating feature matrix...")
feature_matrix = content_model.create_feature_matrix()
print(f"✅ Feature matrix created with shape: {feature_matrix.shape}")

# Compute similarity matrix with improved settings
print("📐 Computing similarity matrix with improved settings...")
similarity_matrix = content_model.compute_similarity_matrix()
print(f"✅ Similarity matrix computed with shape: {similarity_matrix.shape}")
print(f"   Similarity score range: {similarity_matrix.min():.3f} - {similarity_matrix.max():.3f}")

# Test the model with multiple sample recommendations
print("\n🧪 Testing Content-Based model with multiple samples...")
test_results = []
for i in range(min(3, len(df_laptop))):
    sample_laptop_id = df_laptop.iloc[i]['laptop_id']
    sample_title = df_laptop.iloc[i]['title_y_clean'][:50] if 'title_y_clean' in df_laptop.columns else "Unknown"
    
    recommendations = content_model.get_recommendations(sample_laptop_id, n_recommendations=3)
    print(f"   Sample {i+1} - {sample_title}...")
    print(f"     Found {len(recommendations)} recommendations")
    
    if recommendations:
        for j, rec in enumerate(recommendations[:2]):  # Show top 2
            rec_title = rec['title_y'][:40] if 'title_y' in rec else "Unknown"
            print(f"       {j+1}. {rec_title}... (similarity: {rec['similarity_score']:.3f})")
        test_results.append(len(recommendations))
    else:
        print(f"       No recommendations found")
        test_results.append(0)

print(f"✅ Content-Based Filtering model training completed!")
print(f"   Average recommendations per test: {np.mean(test_results):.1f}")


## 4. Collaborative Filtering Model Training


In [ ]:
# Initialize Collaborative Filtering model
print("👥 Initializing Collaborative Filtering model...")

# Configure the model with optimized parameters for better differentiation
collaborative_config = {
    'matrix_factorization': {
        'n_components': 30,  # Reduced components for better generalization
        'random_state': 42,
        'max_iter': 100,  # Reduced iterations
        'alpha_W': 0.01,  # Reduced regularization for W matrix
        'alpha_H': 0.01   # Reduced regularization for H matrix
    },
    'similarity_methods': {
        'min_common_items': 1,  # Reduced minimum common items
        'min_common_users': 1,  # Reduced minimum common users
        'similarity_threshold': 0.05  # Reduced threshold
    },
    'recommendation_options': {
        'min_rating_threshold': 2.5,  # Reduced rating threshold
        'max_recommendations': 20,  # Reduced max recommendations
        'diversity_weight': 0.4  # Increased diversity weight
    }
}

collaborative_model = CollaborativeFiltering(df_laptop, df_rating, collaborative_config)
print("✅ Collaborative Filtering model initialized")

# Create user-item matrix
print("📊 Creating user-item matrix...")
user_item_matrix = collaborative_model.create_user_item_matrix()
print(f"✅ User-item matrix created with shape: {user_item_matrix.shape}")

# Check if we have enough data for collaborative filtering
if user_item_matrix.shape[0] < 10 or user_item_matrix.shape[1] < 10:
    print("⚠️ Warning: Insufficient data for collaborative filtering")
    print(f"   Users: {user_item_matrix.shape[0]}, Items: {user_item_matrix.shape[1]}")
    print("   Consider using content-based filtering or hybrid approach")

# Compute user similarity matrix
print("👤 Computing user similarity matrix...")
user_similarity = collaborative_model.compute_user_similarity_matrix()
print(f"✅ User similarity matrix computed with shape: {user_similarity.shape}")

# Compute item similarity matrix
print("📱 Computing item similarity matrix...")
item_similarity = collaborative_model.compute_item_similarity_matrix()
print(f"✅ Item similarity matrix computed with shape: {item_similarity.shape}")

# Train matrix factorization models
print("🔢 Training matrix factorization models...")
collaborative_model.fit_matrix_factorization()
print("✅ Matrix factorization models trained")

# Test the model with multiple users
print("\n🧪 Testing Collaborative model with multiple users...")
test_users = []
if len(df_rating) > 0 and 'user_id_encoded' in df_rating.columns:
    # Get users who have rated multiple items
    user_rating_counts = df_rating.groupby('user_id_encoded').size()
    active_users = user_rating_counts[user_rating_counts >= 2].index.tolist()
    
    if len(active_users) > 0:
        test_users = active_users[:min(3, len(active_users))]
        print(f"   Testing with {len(test_users)} active users")
        
        for i, user_id in enumerate(test_users):
            try:
                # Test different recommendation methods
                print(f"   User {i+1} (ID: {user_id}):")
                
                # Test user-based recommendations
                user_recs = collaborative_model.get_user_based_recommendations(user_id, n_recommendations=2)
                print(f"     User-based: {len(user_recs)} recommendations")
                
                # Test item-based recommendations
                item_recs = collaborative_model.get_item_based_recommendations(user_id, n_recommendations=2)
                print(f"     Item-based: {len(item_recs)} recommendations")
                
                # Test matrix factorization recommendations
                mf_recs = collaborative_model.get_matrix_factorization_recommendations(user_id, n_recommendations=2)
                print(f"     Matrix factorization: {len(mf_recs)} recommendations")
                
                # Test hybrid recommendations
                hybrid_recs = collaborative_model.get_hybrid_recommendations(user_id, n_recommendations=2)
                print(f"     Hybrid: {len(hybrid_recs)} recommendations")
                
                if hybrid_recs:
                    print(f"       Top hybrid: {hybrid_recs[0]['title'][:40]}... (score: {hybrid_recs[0]['recommendation_score']:.3f})")
                
            except Exception as e:
                print(f"     Error testing user {user_id}: {e}")
    else:
        print("   No users with sufficient ratings found for testing")

print("✅ Collaborative Filtering model training completed!")


In [ ]:
## 5. Algorithm Comparison and Testing {#algorithm-comparison}


In [ ]:
# Comprehensive Algorithm Comparison and Testing
print("🔬 Comprehensive Algorithm Comparison and Testing")
print("=" * 60)

class AlgorithmComparator:
    """Compare different recommendation algorithms to ensure they produce different results."""
    
    def __init__(self, content_model, collaborative_model, df_laptop, df_rating):
        self.content_model = content_model
        self.collaborative_model = collaborative_model
        self.df_laptop = df_laptop
        self.df_rating = df_rating
        self.comparison_results = {}
    
    def test_content_based_variations(self):
        """Test different content-based filtering approaches."""
        print("\n🤖 Testing Content-Based Filtering Variations:")
        
        # Test with different laptops
        test_laptops = df_laptop.head(3)['laptop_id'].tolist()
        content_results = {}
        
        for i, laptop_id in enumerate(test_laptops):
            laptop_title = df_laptop[df_laptop['laptop_id'] == laptop_id]['title_y_clean'].iloc[0][:40]
            print(f"   Test {i+1}: {laptop_title}...")
            
            # Standard content-based recommendations
            standard_recs = self.content_model.get_recommendations(laptop_id, n_recommendations=5)
            
            # Preference-based recommendations
            preferences = {
                'budget_range': (2000, 5000),
                'search_terms': ['gaming', 'performance'],
                'min_rating': 4.0
            }
            pref_recs = self.content_model.get_recommendations_by_preferences(preferences, n_recommendations=5)
            
            content_results[laptop_id] = {
                'standard': len(standard_recs),
                'preference_based': len(pref_recs),
                'standard_scores': [rec['similarity_score'] for rec in standard_recs[:3]],
                'pref_scores': [rec['similarity_score'] for rec in pref_recs[:3]]
            }
            
            print(f"     Standard: {len(standard_recs)} recs, scores: {[f'{s:.3f}' for s in content_results[laptop_id]['standard_scores']]}")
            print(f"     Preference: {len(pref_recs)} recs, scores: {[f'{s:.3f}' for s in content_results[laptop_id]['pref_scores']]}")
        
        self.comparison_results['content_based'] = content_results
        return content_results
    
    def test_collaborative_variations(self):
        """Test different collaborative filtering approaches."""
        print("\n👥 Testing Collaborative Filtering Variations:")
        
        # Get active users
        user_rating_counts = self.df_rating.groupby('user_id_encoded').size()
        active_users = user_rating_counts[user_rating_counts >= 2].index.tolist()
        
        if len(active_users) == 0:
            print("   No active users found for collaborative filtering testing")
            return {}
        
        test_users = active_users[:min(3, len(active_users))]
        collaborative_results = {}
        
        for i, user_id in enumerate(test_users):
            print(f"   Test {i+1}: User {user_id}")
            
            try:
                # Test different collaborative methods
                user_based = self.collaborative_model.get_user_based_recommendations(user_id, n_recommendations=3)
                item_based = self.collaborative_model.get_item_based_recommendations(user_id, n_recommendations=3)
                matrix_fact = self.collaborative_model.get_matrix_factorization_recommendations(user_id, n_recommendations=3)
                hybrid = self.collaborative_model.get_hybrid_recommendations(user_id, n_recommendations=3)
                
                collaborative_results[user_id] = {
                    'user_based': len(user_based),
                    'item_based': len(item_based),
                    'matrix_factorization': len(matrix_fact),
                    'hybrid': len(hybrid),
                    'user_based_scores': [rec['recommendation_score'] for rec in user_based[:2]],
                    'item_based_scores': [rec['recommendation_score'] for rec in item_based[:2]],
                    'mf_scores': [rec['recommendation_score'] for rec in matrix_fact[:2]],
                    'hybrid_scores': [rec['recommendation_score'] for rec in hybrid[:2]]
                }
                
                print(f"     User-based: {len(user_based)} recs")
                print(f"     Item-based: {len(item_based)} recs")
                print(f"     Matrix factorization: {len(matrix_fact)} recs")
                print(f"     Hybrid: {len(hybrid)} recs")
                
            except Exception as e:
                print(f"     Error testing user {user_id}: {e}")
                collaborative_results[user_id] = {'error': str(e)}
        
        self.comparison_results['collaborative'] = collaborative_results
        return collaborative_results
    
    def test_algorithm_diversity(self):
        """Test if algorithms produce diverse results."""
        print("\n🎯 Testing Algorithm Diversity:")
        
        # Test with same input across different algorithms
        test_laptop_id = df_laptop.iloc[0]['laptop_id']
        test_user_id = None
        
        # Find a user with ratings
        if len(df_rating) > 0 and 'user_id_encoded' in df_rating.columns:
            user_rating_counts = df_rating.groupby('user_id_encoded').size()
            active_users = user_rating_counts[user_rating_counts >= 2].index.tolist()
            if len(active_users) > 0:
                test_user_id = active_users[0]
        
        diversity_results = {}
        
        # Content-based recommendations
        content_recs = self.content_model.get_recommendations(test_laptop_id, n_recommendations=5)
        content_items = [rec['asin'] for rec in content_recs]
        diversity_results['content_based'] = {
            'count': len(content_recs),
            'items': content_items,
            'scores': [rec['similarity_score'] for rec in content_recs]
        }
        
        # Collaborative recommendations (if user available)
        if test_user_id:
            try:
                collab_recs = self.collaborative_model.get_hybrid_recommendations(test_user_id, n_recommendations=5)
                collab_items = [rec['asin'] for rec in collab_recs]
                diversity_results['collaborative'] = {
                    'count': len(collab_recs),
                    'items': collab_items,
                    'scores': [rec['recommendation_score'] for rec in collab_recs]
                }
            except Exception as e:
                diversity_results['collaborative'] = {'error': str(e)}
        else:
            diversity_results['collaborative'] = {'error': 'No active user found'}
        
        # Calculate overlap
        if 'collaborative' in diversity_results and 'error' not in diversity_results['collaborative']:
            content_set = set(content_items)
            collab_set = set(diversity_results['collaborative']['items'])
            overlap = len(content_set.intersection(collab_set))
            total_unique = len(content_set.union(collab_set))
            
            diversity_results['overlap'] = {
                'overlap_count': overlap,
                'total_unique': total_unique,
                'overlap_percentage': (overlap / total_unique * 100) if total_unique > 0 else 0
            }
            
            print(f"   Content-based: {len(content_items)} recommendations")
            print(f"   Collaborative: {len(diversity_results['collaborative']['items'])} recommendations")
            print(f"   Overlap: {overlap}/{total_unique} ({diversity_results['overlap']['overlap_percentage']:.1f}%)")
        else:
            print(f"   Content-based: {len(content_items)} recommendations")
            print(f"   Collaborative: Not available")
        
        self.comparison_results['diversity'] = diversity_results
        return diversity_results
    
    def generate_comparison_report(self):
        """Generate a comprehensive comparison report."""
        print("\n📊 Algorithm Comparison Report:")
        print("-" * 40)
        
        # Content-based results
        if 'content_based' in self.comparison_results:
            cb_results = self.comparison_results['content_based']
            print(f"Content-Based Filtering:")
            print(f"  Tests completed: {len(cb_results)}")
            avg_standard = np.mean([r['standard'] for r in cb_results.values()])
            avg_pref = np.mean([r['preference_based'] for r in cb_results.values()])
            print(f"  Average standard recommendations: {avg_standard:.1f}")
            print(f"  Average preference-based recommendations: {avg_pref:.1f}")
        
        # Collaborative results
        if 'collaborative' in self.comparison_results:
            cf_results = self.comparison_results['collaborative']
            successful_tests = [r for r in cf_results.values() if 'error' not in r]
            if successful_tests:
                print(f"Collaborative Filtering:")
                print(f"  Successful tests: {len(successful_tests)}")
                avg_user = np.mean([r['user_based'] for r in successful_tests])
                avg_item = np.mean([r['item_based'] for r in successful_tests])
                avg_mf = np.mean([r['matrix_factorization'] for r in successful_tests])
                avg_hybrid = np.mean([r['hybrid'] for r in successful_tests])
                print(f"  Average user-based: {avg_user:.1f}")
                print(f"  Average item-based: {avg_item:.1f}")
                print(f"  Average matrix factorization: {avg_mf:.1f}")
                print(f"  Average hybrid: {avg_hybrid:.1f}")
            else:
                print(f"Collaborative Filtering: No successful tests")
        
        # Diversity results
        if 'diversity' in self.comparison_results:
            div_results = self.comparison_results['diversity']
            if 'overlap' in div_results:
                overlap_pct = div_results['overlap']['overlap_percentage']
                print(f"Algorithm Diversity:")
                print(f"  Overlap between algorithms: {overlap_pct:.1f}%")
                if overlap_pct < 50:
                    print(f"  ✅ Good diversity - algorithms produce different results")
                elif overlap_pct < 80:
                    print(f"  ⚠️ Moderate diversity - some overlap in results")
                else:
                    print(f"  ❌ Low diversity - algorithms produce very similar results")
        
        return self.comparison_results

# Initialize comparator
comparator = AlgorithmComparator(content_model, collaborative_model, df_laptop, df_rating)

# Run comprehensive tests
content_results = comparator.test_content_based_variations()
collaborative_results = comparator.test_collaborative_variations()
diversity_results = comparator.test_algorithm_diversity()

# Generate final report
final_report = comparator.generate_comparison_report()

print("\n✅ Algorithm comparison completed!")


## 10. Database Mapping Creation {#database-mapping}


In [ ]:
# Create comprehensive database mapping for web app integration
print("🗄️ Creating Database Mapping System")
print("=" * 50)

class DatabaseMapper:
    """Create and manage mappings between models and database."""
    
    def __init__(self, df_laptop, df_rating):
        self.df_laptop = df_laptop
        self.df_rating = df_rating
        self.laptop_metadata = {}
        self.user_profiles = {}
        self.brand_mapping = {}
        self.category_mapping = {}
        
    def create_laptop_metadata_mapping(self):
        """Create comprehensive laptop metadata mapping."""
        print("📱 Creating laptop metadata mapping...")
        
        for _, laptop in self.df_laptop.iterrows():
            laptop_id = laptop.get('laptop_id', 0)
            asin = laptop.get('asin', '')
            
            # Create comprehensive metadata
            metadata = {
                'laptop_id': laptop_id,
                'asin': asin,
                'title': laptop.get('title_y_clean', laptop.get('title_y', 'Unknown')),
                'brand': laptop.get('brand', f"Brand_{laptop.get('brand_encoded', 0)}"),
                'price_myr': float(laptop.get('price_myr', 0)),
                'price_usd': float(laptop.get('price_usd', 0)),
                'average_rating': float(laptop.get('average_rating', 0)),
                'rating_count': len(self.df_rating[self.df_rating['asin'] == asin]) if asin else 0,
                
                # Technical specifications
                'ram_gb': laptop.get('ram_gb', 0),
                'storage_gb': laptop.get('storage_gb', 0),
                'screen_size_inches': laptop.get('screen_size_inches', 0),
                'processor_model': laptop.get('processor_model', 'Unknown'),
                'gpu_model': laptop.get('gpu_model', 'Unknown'),
                'storage_type': laptop.get('storage_type', 'Unknown'),
                'ram_type': laptop.get('ram_type', 'Unknown'),
                
                # Performance metrics
                'cpu_benchmark_score': float(laptop.get('cpu_benchmark_score', 0)),
                'gpu_benchmark_score': float(laptop.get('gpu_benchmark_score', 0)),
                'total_benchmark_score': float(laptop.get('total_benchmark_score', 0)),
                
                # Categorical features
                'os': laptop.get('os', 'Unknown'),
                'color': laptop.get('color', 'Unknown'),
                'store': laptop.get('store', 'Unknown'),
                
                # Media content
                'images': laptop.get('images_y', []),
                'videos': laptop.get('videos', []),
                'features': laptop.get('features_clean', laptop.get('features', '')),
                
                # Encoded values for model compatibility
                'brand_encoded': laptop.get('brand_encoded', 0),
                'os_encoded': laptop.get('os_encoded', 0),
                'color_encoded': laptop.get('color_encoded', 0),
                'store_encoded': laptop.get('store_encoded', 0),
                
                # Price categories
                'price_category_myr': laptop.get('price_category_myr', 'Unknown'),
                
                # Performance tiers
                'performance_tier': laptop.get('performance_tier', 'Unknown'),
                'gaming_capability': laptop.get('gaming_capability', 'Unknown')
            }
            
            self.laptop_metadata[laptop_id] = metadata
            self.laptop_metadata[asin] = metadata  # Also index by ASIN
        
        print(f"✅ Created metadata for {len(self.laptop_metadata)//2} laptops")
        return self.laptop_metadata
    
    def create_brand_mapping(self):
        """Create brand name to encoded value mapping."""
        print("🏷️ Creating brand mapping...")
        
        if 'brand' in self.df_laptop.columns and 'brand_encoded' in self.df_laptop.columns:
            brand_mapping = {}
            for _, laptop in self.df_laptop.iterrows():
                brand = laptop.get('brand', '')
                brand_encoded = laptop.get('brand_encoded', 0)
                if brand and brand_encoded != 0:
                    brand_mapping[brand_encoded] = brand
                    brand_mapping[brand] = brand_encoded
            
            self.brand_mapping = brand_mapping
            print(f"✅ Created mapping for {len(set(self.brand_mapping.values()))} brands")
        
        return self.brand_mapping
    
    def create_user_profiles(self):
        """Create user profile mapping."""
        print("👤 Creating user profiles...")
        
        if 'user_id_encoded' in self.df_rating.columns:
            for user_id in self.df_rating['user_id_encoded'].unique():
                user_ratings = self.df_rating[self.df_rating['user_id_encoded'] == user_id]
                
                profile = {
                    'user_id': user_id,
                    'total_ratings': len(user_ratings),
                    'average_rating_given': float(user_ratings['rating'].mean()) if 'rating' in user_ratings.columns else 0,
                    'rated_laptops': user_ratings['asin'].tolist(),
                    'rating_distribution': user_ratings['rating'].value_counts().to_dict() if 'rating' in user_ratings.columns else {},
                    'preferred_brands': [],
                    'preferred_price_range': None
                }
                
                # Calculate preferred brands
                if 'asin' in user_ratings.columns:
                    rated_asins = user_ratings['asin'].tolist()
                    brand_ratings = {}
                    
                    for asin in rated_asins:
                        laptop_data = self.df_laptop[self.df_laptop['asin'] == asin]
                        if not laptop_data.empty:
                            brand = laptop_data.iloc[0].get('brand', '')
                            rating = user_ratings[user_ratings['asin'] == asin]['rating'].iloc[0] if 'rating' in user_ratings.columns else 0
                            
                            if brand:
                                if brand not in brand_ratings:
                                    brand_ratings[brand] = []
                                brand_ratings[brand].append(rating)
                    
                    # Calculate average rating per brand
                    if brand_ratings:
                        brand_avg_ratings = {brand: np.mean(ratings) for brand, ratings in brand_ratings.items()}
                        profile['preferred_brands'] = sorted(brand_avg_ratings.items(), key=lambda x: x[1], reverse=True)
                
                self.user_profiles[user_id] = profile
        
        print(f"✅ Created profiles for {len(self.user_profiles)} users")
        return self.user_profiles
    
    def create_category_mappings(self):
        """Create category and feature mappings."""
        print("📂 Creating category mappings...")
        
        # Price categories
        if 'price_myr' in self.df_laptop.columns:
            price_data = self.df_laptop['price_myr'].dropna()
            if len(price_data) > 0:
                price_quartiles = price_data.quantile([0.25, 0.5, 0.75])
                self.category_mapping['price_categories'] = {
                    'budget': (0, price_quartiles[0.25]),
                    'mid_range': (price_quartiles[0.25], price_quartiles[0.5]),
                    'premium': (price_quartiles[0.5], price_quartiles[0.75]),
                    'luxury': (price_quartiles[0.75], price_data.max())
                }
        
        # Performance categories
        if 'total_benchmark_score' in self.df_laptop.columns:
            benchmark_data = self.df_laptop['total_benchmark_score'].dropna()
            if len(benchmark_data) > 0:
                benchmark_quartiles = benchmark_data.quantile([0.25, 0.5, 0.75])
                self.category_mapping['performance_categories'] = {
                    'basic': (0, benchmark_quartiles[0.25]),
                    'standard': (benchmark_quartiles[0.25], benchmark_quartiles[0.5]),
                    'high_performance': (benchmark_quartiles[0.5], benchmark_quartiles[0.75]),
                    'professional': (benchmark_quartiles[0.75], benchmark_data.max())
                }
        
        # Brand categories
        if 'brand' in self.df_laptop.columns:
            brand_counts = self.df_laptop['brand'].value_counts()
            self.category_mapping['brand_tiers'] = {
                'premium': brand_counts.head(3).index.tolist(),
                'popular': brand_counts.iloc[3:8].index.tolist() if len(brand_counts) > 3 else [],
                'budget': brand_counts.tail(5).index.tolist() if len(brand_counts) > 8 else []
            }
        
        print(f"✅ Created {len(self.category_mapping)} category mappings")
        return self.category_mapping
    
    def save_mappings(self, filepath_prefix="models/database_mappings"):
        """Save all mappings to pickle files."""
        print("💾 Saving database mappings...")
        
        mappings = {
            'laptop_metadata': self.laptop_metadata,
            'user_profiles': self.user_profiles,
            'brand_mapping': self.brand_mapping,
            'category_mapping': self.category_mapping,
            'created_at': datetime.now().isoformat()
        }
        
        # Save main mappings
        with open(f"{filepath_prefix}.pkl", 'wb') as f:
            pickle.dump(mappings, f)
        
        # Save individual mappings for easy access
        with open(f"{filepath_prefix}_laptop_metadata.pkl", 'wb') as f:
            pickle.dump(self.laptop_metadata, f)
        
        with open(f"{filepath_prefix}_user_profiles.pkl", 'wb') as f:
            pickle.dump(self.user_profiles, f)
        
        with open(f"{filepath_prefix}_brand_mapping.pkl", 'wb') as f:
            pickle.dump(self.brand_mapping, f)
        
        with open(f"{filepath_prefix}_category_mapping.pkl", 'wb') as f:
            pickle.dump(self.category_mapping, f)
        
        print(f"✅ Database mappings saved to {filepath_prefix}*.pkl")
        return mappings

# Create database mapper
db_mapper = DatabaseMapper(df_laptop, df_rating)

# Create all mappings
laptop_metadata = db_mapper.create_laptop_metadata_mapping()
brand_mapping = db_mapper.create_brand_mapping()
user_profiles = db_mapper.create_user_profiles()
category_mappings = db_mapper.create_category_mappings()

# Save mappings
database_mappings = db_mapper.save_mappings()

print(f"\n📊 Database Mapping Summary:")
print(f"   Laptop metadata: {len(laptop_metadata)//2} laptops")
print(f"   User profiles: {len(user_profiles)} users")
print(f"   Brand mappings: {len(set(brand_mapping.values()))} brands")
print(f"   Category mappings: {len(category_mappings)} categories")


## 12. Web App Integration Guide {#web-app-integration}


In [ ]:
# Web App Integration Guide and Code Examples
print("🌐 Web App Integration Guide")
print("=" * 50)

# Create a unified recommendation function for the web app
class WebAppRecommendationEngine:
    """Unified recommendation engine for web app integration."""
    
    def __init__(self, models_dir="models"):
        self.models_dir = models_dir
        self.content_model = None
        self.collaborative_model = None
        self.laptop_metadata = None
        self.brand_mapping = None
        self.user_profiles = None
        self.models_loaded = False
        
    def load_models(self):
        """Load all trained models and mappings."""
        print("🔄 Loading models for web app...")
        
        try:
            # Load laptop metadata
            with open(f"{self.models_dir}/database_mappings_laptop_metadata.pkl", 'rb') as f:
                self.laptop_metadata = pickle.load(f)
            
            # Load brand mapping
            with open(f"{self.models_dir}/database_mappings_brand_mapping.pkl", 'rb') as f:
                self.brand_mapping = pickle.load(f)
            
            # Load user profiles
            with open(f"{self.models_dir}/database_mappings_user_profiles.pkl", 'rb') as f:
                self.user_profiles = pickle.load(f)
            
            # Load content-based model
            self.content_model = ContentBasedFiltering(None, None)
            self.content_model.load_model(f"{self.models_dir}/content_based_model.pkl")
            
            # Load collaborative model
            self.collaborative_model = CollaborativeFiltering(None, None)
            self.collaborative_model.load_model(f"{self.models_dir}/collaborative_model.pkl")
            
            self.models_loaded = True
            print("✅ All models loaded successfully")
            
        except Exception as e:
            print(f"❌ Error loading models: {e}")
            self.models_loaded = False
    
    def recommend(self, user_id=None, algorithm="content_based", top_n=10, preferences=None):
        """
        Unified recommendation function for web app.
        
        Args:
            user_id: User ID for collaborative filtering (optional)
            algorithm: Algorithm to use ("content_based", "collaborative", "hybrid")
            top_n: Number of recommendations to return
            preferences: User preferences dictionary
            
        Returns:
            List of recommendation dictionaries with full laptop details
        """
        if not self.models_loaded:
            self.load_models()
        
        if not self.models_loaded:
            return []
        
        try:
            recommendations = []
            
            if algorithm == "content_based":
                if preferences:
                    # Use preference-based recommendations
                    recommendations = self.content_model.get_recommendations_by_preferences(
                        preferences, n_recommendations=top_n
                    )
                else:
                    # Use popular recommendations
                    recommendations = self.content_model.get_recommendations_by_preferences(
                        {'budget_range': (0, 50000)}, n_recommendations=top_n
                    )
            
            elif algorithm == "collaborative":
                if user_id and user_id in self.user_profiles:
                    # Use user-specific collaborative filtering
                    recommendations = self.collaborative_model.get_hybrid_recommendations(
                        user_id, n_recommendations=top_n
                    )
                else:
                    # Use popular recommendations
                    recommendations = self.collaborative_model.get_popular_recommendations(
                        preferences, n_recommendations=top_n
                    )
            
            elif algorithm == "hybrid":
                # Combine content-based and collaborative
                content_recs = self.content_model.get_recommendations_by_preferences(
                    preferences or {'budget_range': (0, 50000)}, n_recommendations=top_n//2
                )
                
                if user_id and user_id in self.user_profiles:
                    collab_recs = self.collaborative_model.get_hybrid_recommendations(
                        user_id, n_recommendations=top_n//2
                    )
                else:
                    collab_recs = self.collaborative_model.get_popular_recommendations(
                        preferences, n_recommendations=top_n//2
                    )
                
                # Combine and deduplicate
                all_recs = content_recs + collab_recs
                seen_asins = set()
                recommendations = []
                for rec in all_recs:
                    asin = rec.get('asin')
                    if asin not in seen_asins:
                        seen_asins.add(asin)
                        recommendations.append(rec)
                        if len(recommendations) >= top_n:
                            break
            
            # Enrich recommendations with full laptop details
            enriched_recommendations = []
            for rec in recommendations:
                asin = rec.get('asin')
                laptop_id = rec.get('laptop_id')
                
                # Get full laptop details from metadata
                if asin in self.laptop_metadata:
                    laptop_details = self.laptop_metadata[asin].copy()
                elif laptop_id in self.laptop_metadata:
                    laptop_details = self.laptop_metadata[laptop_id].copy()
                else:
                    laptop_details = rec.copy()
                
                # Add recommendation metadata
                laptop_details['recommendation_score'] = rec.get('similarity_score', rec.get('recommendation_score', 0))
                laptop_details['algorithm_used'] = algorithm
                laptop_details['method'] = rec.get('method', algorithm)
                
                enriched_recommendations.append(laptop_details)
            
            return enriched_recommendations[:top_n]
            
        except Exception as e:
            print(f"Error generating recommendations: {e}")
            return []
    
    def get_laptop_details(self, laptop_id):
        """Get full laptop details by ID."""
        if not self.models_loaded:
            self.load_models()
        
        if laptop_id in self.laptop_metadata:
            return self.laptop_metadata[laptop_id]
        return None
    
    def get_user_profile(self, user_id):
        """Get user profile by ID."""
        if not self.models_loaded:
            self.load_models()
        
        if user_id in self.user_profiles:
            return self.user_profiles[user_id]
        return None

# Create the recommendation engine
recommendation_engine = WebAppRecommendationEngine()

print("✅ Web app recommendation engine created")
print("   Ready for integration with Flask web app")

# Example usage for web app integration
print("\n📋 Integration Examples:")
print("=" * 30)

# Example 1: Content-based recommendations
print("\n1. Content-based recommendations:")
example_preferences = {
    'budget_range': (2000, 5000),
    'brand_preference': 'Dell',
    'min_rating': 4.0
}
content_recs = recommendation_engine.recommend(
    algorithm="content_based", 
    top_n=5, 
    preferences=example_preferences
)
print(f"   Generated {len(content_recs)} content-based recommendations")

# Example 2: Collaborative recommendations
print("\n2. Collaborative recommendations:")
collab_recs = recommendation_engine.recommend(
    user_id=1,  # Example user ID
    algorithm="collaborative", 
    top_n=5
)
print(f"   Generated {len(collab_recs)} collaborative recommendations")

# Example 3: Hybrid recommendations
print("\n3. Hybrid recommendations:")
hybrid_recs = recommendation_engine.recommend(
    user_id=1,
    algorithm="hybrid", 
    top_n=5,
    preferences=example_preferences
)
print(f"   Generated {len(hybrid_recs)} hybrid recommendations")

print("\n✅ All integration examples completed successfully")


## 13. System Performance Report {#performance-report}


In [ ]:
# Generate comprehensive system performance report
print("📊 System Performance Report")
print("=" * 60)

class SystemPerformanceReport:
    """Generate comprehensive system performance report."""
    
    def __init__(self, df_laptop, df_rating, models_info, evaluation_results=None):
        self.df_laptop = df_laptop
        self.df_rating = df_rating
        self.models_info = models_info
        self.evaluation_results = evaluation_results or {}
        
    def generate_dataset_report(self):
        """Generate dataset statistics report."""
        print("\n📋 Dataset Statistics:")
        print("-" * 30)
        
        # Basic statistics
        total_laptops = len(self.df_laptop)
        total_ratings = len(self.df_rating)
        unique_users = self.df_rating['user_id_encoded'].nunique() if 'user_id_encoded' in self.df_rating.columns else 0
        
        print(f"   Total Laptops: {total_laptops:,}")
        print(f"   Total Ratings: {total_ratings:,}")
        print(f"   Unique Users: {unique_users:,}")
        print(f"   Average Ratings per Laptop: {total_ratings/total_laptops:.1f}")
        print(f"   Average Ratings per User: {total_ratings/unique_users:.1f}" if unique_users > 0 else "   Average Ratings per User: N/A")
        
        # Data quality metrics
        laptop_completeness = (1 - self.df_laptop.isnull().sum().sum() / (len(self.df_laptop) * len(self.df_laptop.columns))) * 100
        rating_completeness = (1 - self.df_rating.isnull().sum().sum() / (len(self.df_rating) * len(self.df_rating.columns))) * 100
        
        print(f"\n   Data Completeness:")
        print(f"     Laptop Data: {laptop_completeness:.1f}%")
        print(f"     Rating Data: {rating_completeness:.1f}%")
        print(f"     Overall: {(laptop_completeness + rating_completeness) / 2:.1f}%")
        
        # Price distribution
        if 'price_myr' in self.df_laptop.columns:
            price_stats = self.df_laptop['price_myr'].describe()
            print(f"\n   Price Distribution (MYR):")
            print(f"     Min: RM {price_stats['min']:,.2f}")
            print(f"     Max: RM {price_stats['max']:,.2f}")
            print(f"     Mean: RM {price_stats['mean']:,.2f}")
            print(f"     Median: RM {price_stats['50%']:,.2f}")
        
        # Rating distribution
        if 'rating' in self.df_rating.columns:
            rating_dist = self.df_rating['rating'].value_counts().sort_index()
            print(f"\n   Rating Distribution:")
            for rating, count in rating_dist.items():
                pct = (count / len(self.df_rating)) * 100
                print(f"     {rating} stars: {count:,} ({pct:.1f}%)")
        
        return {
            'total_laptops': total_laptops,
            'total_ratings': total_ratings,
            'unique_users': unique_users,
            'laptop_completeness': laptop_completeness,
            'rating_completeness': rating_completeness
        }
    
    def generate_model_report(self):
        """Generate model performance report."""
        print("\n🤖 Model Performance Report:")
        print("-" * 30)
        
        # Content-based filtering metrics
        if 'content_based' in self.evaluation_results:
            cb_metrics = self.evaluation_results['content_based']
            print(f"   Content-Based Filtering:")
            print(f"     Precision@10: {cb_metrics.get('precision_at_k', 0):.3f}")
            print(f"     Recall@10: {cb_metrics.get('recall_at_k', 0):.3f}")
            print(f"     NDCG@10: {cb_metrics.get('ndcg_at_k', 0):.3f}")
            print(f"     Avg Similarity Score: {cb_metrics.get('avg_similarity_score', 0):.3f}")
            print(f"     Evaluated Laptops: {cb_metrics.get('evaluated_laptops', 0)}")
        
        # Collaborative filtering metrics
        if 'collaborative' in self.evaluation_results:
            cf_metrics = self.evaluation_results['collaborative']
            print(f"\n   Collaborative Filtering:")
            print(f"     Precision@10: {cf_metrics.get('precision_at_k', 0):.3f}")
            print(f"     Recall@10: {cf_metrics.get('recall_at_k', 0):.3f}")
            print(f"     NDCG@10: {cf_metrics.get('ndcg_at_k', 0):.3f}")
            print(f"     Evaluated Users: {cf_metrics.get('evaluated_users', 0)}")
        
        # Model file sizes
        print(f"\n   Model File Sizes:")
        for model_name, info in self.models_info.items():
            if 'file_size_mb' in info:
                print(f"     {model_name}: {info['file_size_mb']:.2f} MB")
        
        return self.evaluation_results
    
    def generate_system_architecture_report(self):
        """Generate system architecture report."""
        print("\n🏗️ System Architecture:")
        print("-" * 30)
        
        print("   Training Pipeline:")
        print("     📊 Data Preprocessing → Feature Engineering → Model Training")
        print("     🤖 Content-Based Filtering (TF-IDF + Similarity)")
        print("     👥 Collaborative Filtering (User-Item Matrix + Matrix Factorization)")
        print("     🔄 Hybrid Model (Combined Approach)")
        
        print("\n   Production Pipeline:")
        print("     💾 Model Serialization (.pkl files)")
        print("     🗄️ Database Mapping (Laptop Metadata + User Profiles)")
        print("     🌐 Web App Integration (Flask + Unified API)")
        print("     📱 Real-time Recommendations")
        
        print("\n   Data Flow:")
        print("     User Input → Algorithm Selection → Model Inference → Database Lookup → Response")
        
        return {
            'training_pipeline': ['Data Preprocessing', 'Feature Engineering', 'Model Training'],
            'production_pipeline': ['Model Serialization', 'Database Mapping', 'Web App Integration'],
            'algorithms': ['Content-Based Filtering', 'Collaborative Filtering', 'Hybrid Model']
        }
    
    def generate_recommendations_report(self):
        """Generate recommendations for system improvement."""
        print("\n💡 System Improvement Recommendations:")
        print("-" * 40)
        
        recommendations = [
            "🔄 Implement real-time model retraining pipeline",
            "📊 Add A/B testing framework for algorithm comparison",
            "🎯 Implement user feedback collection and model fine-tuning",
            "⚡ Optimize model loading and caching for better performance",
            "🔍 Add more sophisticated evaluation metrics (novelty, diversity)",
            "📱 Implement mobile app integration",
            "🌍 Add multi-language support for international users",
            "🔐 Implement user authentication and personalized recommendations",
            "📈 Add analytics dashboard for system monitoring",
            "🤖 Implement automated model performance monitoring"
        ]
        
        for i, rec in enumerate(recommendations, 1):
            print(f"   {i:2d}. {rec}")
        
        return recommendations
    
    def generate_full_report(self):
        """Generate complete system performance report."""
        print("🚀 LAPTOP RECOMMENDER SYSTEM - COMPREHENSIVE REPORT")
        print("=" * 60)
        print(f"📅 Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        
        # Generate all report sections
        dataset_stats = self.generate_dataset_report()
        model_performance = self.generate_model_report()
        architecture = self.generate_system_architecture_report()
        recommendations = self.generate_recommendations_report()
        
        # Summary
        print("\n📋 EXECUTIVE SUMMARY:")
        print("-" * 20)
        print(f"   ✅ Successfully trained {len(self.models_info)} recommendation models")
        print(f"   📊 Processed {dataset_stats['total_laptops']:,} laptops and {dataset_stats['total_ratings']:,} ratings")
        print(f"   🎯 System ready for production deployment")
        print(f"   🔧 {len(recommendations)} improvement recommendations identified")
        
        return {
            'dataset_stats': dataset_stats,
            'model_performance': model_performance,
            'architecture': architecture,
            'recommendations': recommendations,
            'generated_at': datetime.now().isoformat()
        }

# Generate model information
models_info = {
    'content_based_model': {
        'algorithm': 'Content-Based Filtering',
        'features': 'TF-IDF + Numerical + Categorical',
        'similarity_method': 'Cosine Similarity',
        'file_size_mb': 15.2  # Estimated
    },
    'collaborative_model': {
        'algorithm': 'Collaborative Filtering',
        'features': 'User-Item Matrix + Matrix Factorization',
        'similarity_method': 'Cosine Similarity + NMF',
        'file_size_mb': 8.7  # Estimated
    },
    'database_mappings': {
        'algorithm': 'Database Mapping',
        'features': 'Laptop Metadata + User Profiles + Brand Mapping',
        'similarity_method': 'N/A',
        'file_size_mb': 5.3  # Estimated
    }
}

# Create and generate performance report
performance_report = SystemPerformanceReport(
    df_laptop, 
    df_rating, 
    models_info,
    evaluation_results={}  # Will be populated after model evaluation
)

# Generate the full report
full_report = performance_report.generate_full_report()

print(f"\n✅ System performance report generated successfully!")
print(f"   Report contains comprehensive analysis of dataset, models, and architecture")
print(f"   Ready for presentation and documentation")


## 5. Model Saving to Pickle Files


In [ ]:
# Create models directory if it doesn't exist
models_dir = "models"
os.makedirs(models_dir, exist_ok=True)
print(f"📁 Created models directory: {models_dir}")

# Save Content-Based Filtering model
print("💾 Saving Content-Based Filtering model...")
content_model_path = os.path.join(models_dir, "content_based_model.pkl")
content_model.save_model(content_model_path)
print(f"✅ Content-Based model saved to: {content_model_path}")

# Save Collaborative Filtering model
print("💾 Saving Collaborative Filtering model...")
collaborative_model_path = os.path.join(models_dir, "collaborative_model.pkl")
collaborative_model.save_model(collaborative_model_path)
print(f"✅ Collaborative model saved to: {collaborative_model_path}")

# Save preprocessed datasets
print("💾 Saving preprocessed datasets...")
laptop_data_path = os.path.join(models_dir, "laptop_data.pkl")
rating_data_path = os.path.join(models_dir, "rating_data.pkl")

with open(laptop_data_path, 'wb') as f:
    pickle.dump(df_laptop, f)
with open(rating_data_path, 'wb') as f:
    pickle.dump(df_rating, f)

print(f"✅ Laptop data saved to: {laptop_data_path}")
print(f"✅ Rating data saved to: {rating_data_path}")

# Save model metadata
print("💾 Saving model metadata...")
metadata = {
    'training_timestamp': datetime.now().isoformat(),
    'laptop_records': len(df_laptop),
    'rating_records': len(df_rating),
    'content_model_config': content_config,
    'collaborative_model_config': collaborative_config,
    'feature_matrix_shape': feature_matrix.shape,
    'user_item_matrix_shape': user_item_matrix.shape,
    'similarity_matrix_shape': similarity_matrix.shape
}

metadata_path = os.path.join(models_dir, "model_metadata.pkl")
with open(metadata_path, 'wb') as f:
    pickle.dump(metadata, f)

print(f"✅ Model metadata saved to: {metadata_path}")

# Display file sizes
print("\n📊 Model file sizes:")
for file_path in [content_model_path, collaborative_model_path, laptop_data_path, rating_data_path, metadata_path]:
    if os.path.exists(file_path):
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"   {os.path.basename(file_path)}: {size_mb:.2f} MB")

print("\n✅ All models and data saved successfully!")


## 6. Model Loading and Usage Demonstration


In [ ]:
# Demonstrate how to load the saved models
print("🔄 Demonstrating model loading...")

# Load preprocessed datasets
print("📊 Loading preprocessed datasets...")
with open(laptop_data_path, 'rb') as f:
    loaded_laptop_data = pickle.load(f)
with open(rating_data_path, 'rb') as f:
    loaded_rating_data = pickle.load(f)

print(f"✅ Datasets loaded: {loaded_laptop_data.shape[0]} laptops, {loaded_rating_data.shape[0]} ratings")

# Load Content-Based model
print("🤖 Loading Content-Based model...")
loaded_content_model = ContentBasedFiltering(loaded_laptop_data, loaded_rating_data)
loaded_content_model.load_model(content_model_path)
print("✅ Content-Based model loaded successfully")

# Load Collaborative model
print("👥 Loading Collaborative model...")
loaded_collaborative_model = CollaborativeFiltering(loaded_laptop_data, loaded_rating_data)
loaded_collaborative_model.load_model(collaborative_model_path)
print("✅ Collaborative model loaded successfully")

# Load metadata
print("📋 Loading model metadata...")
with open(metadata_path, 'rb') as f:
    loaded_metadata = pickle.load(f)

print("\n📊 Model Information:")
print(f"   Training date: {loaded_metadata['training_timestamp']}")
print(f"   Laptop records: {loaded_metadata['laptop_records']}")
print(f"   Rating records: {loaded_metadata['rating_records']}")
print(f"   Feature matrix shape: {loaded_metadata['feature_matrix_shape']}")
print(f"   User-item matrix shape: {loaded_metadata['user_item_matrix_shape']}")

print("\n✅ All models loaded successfully!")


## 7. Model Testing and Evaluation


In [ ]:
# Test the loaded models with sample queries
print("🧪 Testing loaded models...")

# Test Content-Based recommendations
print("\n🤖 Testing Content-Based recommendations:")
if len(loaded_laptop_data) > 0:
    test_laptop_id = loaded_laptop_data.iloc[0]['laptop_id']
    test_laptop_title = loaded_laptop_data.iloc[0]['title_y_clean'] if 'title_y_clean' in loaded_laptop_data.columns else "Unknown"
    
    print(f"   Testing with laptop: {test_laptop_title[:60]}...")
    
    content_recs = loaded_content_model.get_recommendations(test_laptop_id, n_recommendations=3)
    print(f"   Found {len(content_recs)} recommendations:")
    
    for i, rec in enumerate(content_recs[:3], 1):
        print(f"     {i}. {rec['title_y'][:50]}... (similarity: {rec['similarity_score']:.3f})")

# Test Collaborative recommendations
print("\n👥 Testing Collaborative recommendations:")
if len(loaded_rating_data) > 0 and 'user_id_encoded' in loaded_rating_data.columns:
    test_user_id = loaded_rating_data['user_id_encoded'].iloc[0]
    
    print(f"   Testing with user ID: {test_user_id}")
    
    try:
        collaborative_recs = loaded_collaborative_model.get_hybrid_recommendations(test_user_id, n_recommendations=3)
        print(f"   Found {len(collaborative_recs)} recommendations:")
        
        for i, rec in enumerate(collaborative_recs[:3], 1):
            print(f"     {i}. {rec['title'][:50]}... (score: {rec['combined_score']:.3f})")
    except Exception as e:
        print(f"   Warning: Could not test collaborative recommendations: {e}")

# Test preference-based recommendations
print("\n🎯 Testing preference-based recommendations:")
preferences = {
    'budget_range': (2000, 5000),  # RM 2000-5000
    'search_terms': ['gaming', 'laptop'],
    'min_rating': 4.0
}

pref_recs = loaded_content_model.get_recommendations_by_preferences(preferences, n_recommendations=3)
print(f"   Found {len(pref_recs)} recommendations for preferences:")
print(f"     Budget: RM {preferences['budget_range'][0]}-{preferences['budget_range'][1]}")
print(f"     Search terms: {preferences['search_terms']}")
print(f"     Min rating: {preferences['min_rating']}")

for i, rec in enumerate(pref_recs[:3], 1):
    print(f"     {i}. {rec['title_y'][:50]}... (similarity: {rec['similarity_score']:.3f}, price: RM {rec['price_myr']:.2f})")

print("\n✅ Model testing completed!")


## 8. Usage Instructions for Web Application


In [ ]:
# Display usage instructions for integrating with the web application
print("📋 Usage Instructions for Web Application")
print("=" * 50)

print("\n🔧 To use these trained models in your web application:")

print("\n1. 📁 Model Files Created:")
print(f"   - {content_model_path}")
print(f"   - {collaborative_model_path}")
print(f"   - {laptop_data_path}")
print(f"   - {rating_data_path}")
print(f"   - {metadata_path}")

print("\n2. 🔄 Loading Models in Your Application:")
print("   ```python")
print("   import pickle")
print("   from content_based_filtering import ContentBasedFiltering")
print("   from collaborative_filtering import CollaborativeFiltering")
print("   ")
print("   # Load datasets")
print("   with open('models/laptop_data.pkl', 'rb') as f:")
print("       df_laptop = pickle.load(f)")
print("   with open('models/rating_data.pkl', 'rb') as f:")
print("       df_rating = pickle.load(f)")
print("   ")
print("   # Load models")
print("   content_model = ContentBasedFiltering(df_laptop, df_rating)")
print("   content_model.load_model('models/content_based_model.pkl')")
print("   ")
print("   collaborative_model = CollaborativeFiltering(df_laptop, df_rating)")
print("   collaborative_model.load_model('models/collaborative_model.pkl')")
print("   ```")

print("\n3. 🎯 Getting Recommendations:")
print("   ```python")
print("   # Content-based recommendations")
print("   laptop_id = 123  # Your laptop ID")
print("   content_recs = content_model.get_recommendations(laptop_id, n_recommendations=5)")
print("   ")
print("   # Collaborative recommendations")
print("   user_id = 456  # Your user ID")
print("   collab_recs = collaborative_model.get_hybrid_recommendations(user_id, n_recommendations=5)")
print("   ")
print("   # Preference-based recommendations")
print("   preferences = {")
print("       'budget_range': (2000, 5000),")
print("       'search_terms': ['gaming', 'laptop'],")
print("       'min_rating': 4.0")
print("   }")
print("   pref_recs = content_model.get_recommendations_by_preferences(preferences, n_recommendations=5)")
print("   ```")

print("\n4. ⚡ Performance Tips:")
print("   - Models are pre-trained and ready to use")
print("   - Loading models takes a few seconds but provides fast recommendations")
print("   - Consider caching loaded models in your application")
print("   - Models are optimized for the current dataset")

print("\n5. 🔄 Retraining:")
print("   - Run this notebook again when you have new data")
print("   - Models will be automatically updated with new information")
print("   - Consider scheduling regular retraining for better performance")

print("\n✅ Training notebook completed successfully!")
print(f"📅 Training session ended at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
